# Test-model

We will load the trained model and test it on the test storms using ACE preliminary parameters.

We will also plot the storms that we will use in the paper and save the predictions to a csv format to get the metrics in the next notebook

In [1]:
# External Libraries
import numpy as np
import torch
import torchinfo
import matplotlib.pyplot as plt
import pandas as pd
import os
from IPython.display import display
import time
import matplotlib.dates as mdates
from IPython.display import display, HTML
import pickle

# Local Libraries
import reldi_modules
import utils
import pre_processing
import storm_dates
import metrics
import constants

# configs
display(HTML("<style>.container { width:100% !important; }</style>"))
pd.set_option("display.max_columns", None)

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA is available. Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("CUDA is not available. Using CPU.")

/mnt/data/ldi-forecast-1and2-hours/ldi-torch/lib/python3.12/site-packages/tslearn/bases/bases.py:15: UserWarning: h5py not installed, hdf5 features will not be supported.
Install h5py to use hdf5 features: http://docs.h5py.org/
  warn(h5py_msg)


CUDA is available. Using GPU: NVIDIA GeForce RTX 3090


In [2]:
LOOKBACK = 12 * 8
NUM_OUTPUTS_MODEL = 3
QUANTILES = [0.05, 0.95]
BATCH_SIZE = 256
D_MODEL = 128
FORECAST_STEPS = 2

indices_cols = ["SYM_H"]
ldi_cols = []
for station in constants.STATIONS:
    indices_cols.append(f"LDi_{station}")
    ldi_cols.append(f"LDi_{station}")

COLS_TO_USE = [
    constants.B_MAGNITUDE_COL_NAME,
    constants.BZ_COL_NAME,
    constants.PROTON_DENSITY_COL_NAME,
    constants.PROTON_SPEED_COL_NAME,
    constants.PROTON_TEMPERATURE_COL_NAME,
    constants.PRESSURE_COL_NAME,
]

COLS_TO_USE_NO_DERIVED = constants.COLS_TO_USE_NO_DERIVED.copy()

COLS_TO_SCALE_LOG = [
    constants.PROTON_DENSITY_COL_NAME,
    constants.PROTON_TEMPERATURE_COL_NAME,
    constants.PROTON_SPEED_COL_NAME,
    constants.PRESSURE_COL_NAME,
]

COLS_TO_SCALE_STANDARD = [constants.PROTON_SPEED_COL_NAME, constants.BZ_COL_NAME]

COLS_TO_SCALE_ROBUST = [
    constants.PROTON_TEMPERATURE_COL_NAME,
    constants.PROTON_DENSITY_COL_NAME,
    constants.PRESSURE_COL_NAME,
    constants.B_MAGNITUDE_COL_NAME,
]


stations_data_path = "./data/stations_data_magnetic.csv"
stations_data = pd.read_csv(stations_data_path)

min_lat = 0
max_lat = 90

min_lon = -180
max_lon = 180


SHIFT_NAMES = []
SHIFT_VALUES = []
SHIFT_FORWARD = []

for i in range(FORECAST_STEPS):
    SHIFT_VALUES.append(-(i + 1) * 12)
    SHIFT_FORWARD.append((i + 1) * 12)
    SHIFT_NAMES.append(f"-{i + 1}h")

SHIFT_HOUR = 12
THRESHOLD_PLOT = -200

NUM_FEATS_SOLAR_WIND = len(COLS_TO_USE)

rotation_columns = []
NUM_FEATS_LDI = 5
NUM_FEATS_POSITION = 4
LABEL_COLUMNS = []
COLS_PLOT = COLS_TO_USE.copy()
INDEX_COLUMNS = []
station_columns_decoder = {}
ldi_columns_decoder = {}
station_labels = {}
station_columns = {}
station_first_ldi = {}
total_station_columns = {}
station_to_labels = {}

for station in constants.STATIONS:
    station_to_labels[station] = []
    station_first_ldi[station] = f"LDi_{station}"
    station_columns[station] = [
        f"LDi_{station}",
        f"MLT_{station}_sin",
        f"MLT_{station}_cos",
        f"Latitude_{station}",
        f"Longitude_{station}",
    ]
    labels_station = []
    station_columns_decoder[station] = []
    for shift_suffix in SHIFT_NAMES:
        labels_station.append(f"LDi_{station}{shift_suffix}")
        station_columns_decoder[station].append(f"MLT_{station}_sin{shift_suffix}")
        station_columns_decoder[station].append(f"MLT_{station}_cos{shift_suffix}")
        station_columns_decoder[station].append(f"Latitude_{station}{shift_suffix}")
        station_columns_decoder[station].append(f"Longitude_{station}{shift_suffix}")

    station_labels[station] = labels_station
    ldi_columns_decoder[station] = [f"LDi_{station}"]
    for shift_suffix in SHIFT_NAMES[:-1]:
        ldi_columns_decoder[station].append(f"LDi_{station}{shift_suffix}")

    rotation_columns.append(f"MLT_{station}_sin")
    rotation_columns.append(f"MLT_{station}_cos")
    rotation_columns.append(f"Latitude_{station}")
    rotation_columns.append(f"Longitude_{station}")
    for i in range(FORECAST_STEPS):
        LABEL_COLUMNS.append(f"LDi_{station}-{i + 1}h")
        station_to_labels[station].append(f"LDi_{station}-{i + 1}h")
    INDEX_COLUMNS.append(f"LDi_{station}")
    COLS_PLOT.append(f"LDi_{station}")
    total_station_columns[station] = (
        station_columns[station].copy() + COLS_TO_USE.copy()
    )


def inside_quantile(row, label, q_lower, q_upper):
    return row[q_lower] <= row[label] <= row[q_upper]


for station in constants.STATIONS:
    os.makedirs(f"figs/{station}", exist_ok=True)
    os.makedirs(f"figs_colors/{station}", exist_ok=True)
    os.makedirs(f"predictions/{station}", exist_ok=True)
    os.makedirs(f"figs/{station}/bfe/", exist_ok=True)
    os.makedirs(f"figs_colors/{station}/bfe/", exist_ok=True)
    os.makedirs(f"figs/{station}/forecasts/", exist_ok=True)
    os.makedirs(f"figs_colors/{station}/forecasts/", exist_ok=True)


SAVE_FIGS = True

In [3]:
all_timeline_data_path = "./data/all_timeline"

ldi_paths = [
    "./data/LDI_ABG.csv",
    "./data/LDI_MMB.csv",
    "./data/LDI_CLF.csv",
    "./data/LDI_TUC.csv",
    "./data/LDI_HON.csv",
    "./data/LDI_SFS.csv",
]

In [4]:
model = torch.load("reldi_1to2h_full_model.pt", weights_only=False).to(device)
model.eval()

solar_wind_input_example = torch.randn(32, LOOKBACK, len(COLS_TO_USE)).to(device)
ldi_input_example = torch.randn(32, LOOKBACK, NUM_FEATS_LDI).to(device)
decoder_input_example = torch.randn(32, FORECAST_STEPS, NUM_FEATS_LDI).to(device)
decoder_input_inference_position_example = torch.randn(
    32, FORECAST_STEPS, NUM_FEATS_POSITION
).to(device)
decoder_input_inference_ldi_example = torch.randn(32, 1, 1).to(device)

print(f"{solar_wind_input_example.shape = }")
print(f"{ldi_input_example.shape = }")
print(f"{decoder_input_example.shape = }")
print(f"{decoder_input_inference_position_example.shape = }")
print(f"{decoder_input_inference_ldi_example.shape = }")

# Teacher forcing
(
    output_teacher_forcing,
    quantile_lower_teacher_forcing,
    quantile_upper_teacher_forcing,
) = model(solar_wind_input_example, ldi_input_example, decoder_input_example)
print(f"{output_teacher_forcing.shape = }")
print(f"{quantile_lower_teacher_forcing.shape = }")
print(f"{quantile_upper_teacher_forcing.shape = }")

# Inference
output_inference, quantile_lower_inference, quantile_upper_inference = model.inference(
    solar_wind_input_example,
    ldi_input_example,
    decoder_input_inference_ldi_example,
    decoder_input_inference_position_example,
)
print(f"{output_inference.shape = }")
print(f"{quantile_lower_inference.shape = }")
print(f"{quantile_upper_inference.shape = }")


print(
    torchinfo.summary(
        model,
        input_data=(solar_wind_input_example, ldi_input_example, decoder_input_example),
    )
)

solar_wind_input_example.shape = torch.Size([32, 96, 6])
ldi_input_example.shape = torch.Size([32, 96, 5])
decoder_input_example.shape = torch.Size([32, 2, 5])
decoder_input_inference_position_example.shape = torch.Size([32, 2, 4])
decoder_input_inference_ldi_example.shape = torch.Size([32, 1, 1])
output_teacher_forcing.shape = torch.Size([32, 2, 1])
quantile_lower_teacher_forcing.shape = torch.Size([32, 2, 1])
quantile_upper_teacher_forcing.shape = torch.Size([32, 2, 1])
output_inference.shape = torch.Size([32, 2, 1])
quantile_lower_inference.shape = torch.Size([32, 2, 1])
quantile_upper_inference.shape = torch.Size([32, 2, 1])
Layer (type:depth-idx)                   Output Shape              Param #
RELDi                                    [32, 2, 1]                --
├─Encoder: 1-1                           [32, 2, 128]              --
│    └─Sequential: 2-1                   [32, 128, 96]             --
│    │    └─CausalConv1d: 3-1            [32, 128, 96]             5,504
│    

/mnt/data/ldi-forecast-1and2-hours/ldi-torch/lib/python3.12/site-packages/torchinfo/torchinfo.py:477: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  action_fn=lambda data: sys.getsizeof(data.storage()),


In [5]:
all_df_ace_imf_provisional = utils.read_data(
    all_timeline_data_path, pattern_to_read=["ace_imf_provisonal_5_min"]
)

all_df_ace_imf_provisional = pd.concat(all_df_ace_imf_provisional)

test_dfs_ace_imf = []

offset = pd.DateOffset(hours=24)

for start_date, end_date, storm_index in storm_dates.TEST_STORMS:
    sd = pd.to_datetime(start_date) - offset
    ed = pd.to_datetime(end_date) + offset
    storm = all_df_ace_imf_provisional[sd:ed].copy()
    storm = pre_processing.preprocess_ace_imf_provisional(storm)
    test_dfs_ace_imf.append(storm.copy())

del all_df_ace_imf_provisional

In [6]:
all_df_ace_swepam_provisional = utils.read_data(
    all_timeline_data_path, pattern_to_read=["ace_swepam_provisional_5_min"]
)

all_df_ace_swepam_provisional = pd.concat(all_df_ace_swepam_provisional)

test_dfs_ace_swepam = []

for start_date, end_date, storm_index in storm_dates.TEST_STORMS:
    sd = pd.to_datetime(start_date) - offset
    ed = pd.to_datetime(end_date) + offset
    storm = all_df_ace_swepam_provisional[sd:ed].copy()
    storm = pre_processing.preprocess_ace_swepam_provisional(storm)
    test_dfs_ace_swepam.append(storm.copy())

del all_df_ace_swepam_provisional

In [7]:
all_omni = utils.read_data(all_timeline_data_path, pattern_to_read=["omni"])
all_omni = pd.concat(all_omni)

In [8]:
ldi = [
    pd.read_csv(ldi_path, parse_dates=["datetime"], index_col="datetime")
    for ldi_path in ldi_paths
]
ldi = pd.concat(ldi, axis=1)

for station in constants.STATIONS:
    ldi[f"Longitude_{station}"] = stations_data[stations_data["IAGA_CODE"] == station][
        "LONGITUDE_M"
    ].values[0]
    ldi[f"Latitude_{station}"] = stations_data[stations_data["IAGA_CODE"] == station][
        "LATITUDE_M"
    ].values[0]

In [9]:
test_storms = []

for i in range(len(test_dfs_ace_imf)):
    storm = test_dfs_ace_imf[i].join(test_dfs_ace_swepam[i])
    storm = storm.join(ldi)
    test_storms.append(storm)

In [10]:
medians = {
    "Bmag": 6.447526,
    "Bz": -0.270270,
    "Proton_density": 4.700450,
    "Proton_speed": 456.152500,
    "Proton_temp": 75410.697500,
    "Pressure": 1.984565,
}

In [11]:
for i, _ in enumerate(test_storms):
    test_storms[i].loc[:, COLS_TO_USE_NO_DERIVED] = (
        test_storms[i].loc[:, COLS_TO_USE_NO_DERIVED].interpolate()
    )
    test_storms[i].loc[:, COLS_TO_USE_NO_DERIVED] = (
        test_storms[i].loc[:, COLS_TO_USE_NO_DERIVED].ffill()
    )
    test_storms[i].loc[:, COLS_TO_USE_NO_DERIVED] = (
        test_storms[i].loc[:, COLS_TO_USE_NO_DERIVED].bfill()
    )
    test_storms[i].loc[:, COLS_TO_USE_NO_DERIVED] = (
        test_storms[i].loc[:, COLS_TO_USE_NO_DERIVED].fillna(medians)
    )

for i, storm in enumerate(test_storms):
    test_storms[i] = pre_processing.calculate_derived_params(storm)

for i, storm in enumerate(test_storms):
    test_storms[i] = test_storms[i].astype("float32")

In [12]:
blacklist_storms = {
    "ABG": [93, 103, 109],
    "MMB": [93, 94],
    "CLF": [],
    "TUC": [],
    "HON": [],
    "SFS": [95],
}

blacklist_dates = {
    "ABG": [],
    "MMB": [],
    "CLF": [],
    "TUC": [
        (
            "2023-03-28 00:00:00",
            "2023-04-01 00:00:00",
        ),  # Storm 96 bunch of NAs at the end
    ],
    "HON": [
        ("2024-06-30 00:00:00", "2024-07-05 00:00:00"),
    ],
    "SFS": [],
}

for start_date, end_date, index in storm_dates.TEST_STORMS:
    for station, blacklist in blacklist_storms.items():
        if index in blacklist:
            for storm_ind in range(len(test_storms)):
                if pd.to_datetime(start_date) in test_storms[storm_ind].index:
                    print(f"Blacklisting test storm {index} for station {station}")
                    df_tmp = test_storms[storm_ind].copy()
                    df_tmp[f"LDi_{station}"] = np.nan
                    test_storms[storm_ind] = df_tmp


for station, dates in blacklist_dates.items():
    for invalid_date in dates:
        if type(invalid_date) is tuple:
            start_invalid = pd.to_datetime(invalid_date[0])
            end_invalid = pd.to_datetime(invalid_date[1])

            for storm_ind in range(len(test_storms)):
                if (
                    start_invalid in test_storms[storm_ind].index
                    or end_invalid in test_storms[storm_ind].index
                ):
                    print(
                        f"Invalidating testing tuple for {station}",
                        start_invalid,
                        end_invalid,
                    )
                    df_tmp = test_storms[storm_ind].copy()
                    df_tmp.loc[start_invalid:end_invalid, f"LDi_{station}"] = np.nan
                    df_tmp[f"LDi_{station}"] = df_tmp[f"LDi_{station}"].interpolate()
                    test_storms[storm_ind] = df_tmp

        else:
            invalid_d = pd.to_datetime(invalid_date)
            print(f"Invalidating date for {station}", invalid_d)

            for storm_ind in range(len(test_storms)):
                if invalid_d in test_storms[storm_ind].index:
                    print(f"Invalidating testing date for {station}", invalid_d)
                    df_tmp = test_storms[storm_ind].copy()
                    df_tmp.loc[invalid_d, f"LDi_{station}"] = np.nan
                    df_tmp[f"LDi_{station}"] = df_tmp[f"LDi_{station}"].interpolate()
                    test_storms[storm_ind] = df_tmp

Blacklisting test storm 93 for station ABG
Blacklisting test storm 93 for station MMB
Blacklisting test storm 94 for station MMB
Blacklisting test storm 95 for station SFS
Blacklisting test storm 103 for station ABG
Blacklisting test storm 109 for station ABG
Invalidating testing tuple for TUC 2023-03-28 00:00:00 2023-04-01 00:00:00
Invalidating testing tuple for HON 2024-06-30 00:00:00 2024-07-05 00:00:00


In [13]:
def normalize_latitude(values, min_value=min_lat, max_value=max_lat):
    return [
        2 * (np.abs(value) - min_value) / (max_value - min_value) - 1
        for value in values
    ]


def normalize_longitude(values, min_value=min_lon, max_value=max_lon):
    return [
        2 * (np.abs(value) - min_value) / (max_value - min_value) - 1
        for value in values
    ]


def encode_MLT(mlt):
    mlt_sin = np.sin(2 * np.pi * mlt / 24)
    mlt_cos = np.cos(2 * np.pi * mlt / 24)
    return np.array([mlt_sin, mlt_cos])


# Save the scalers to the experiments folder
with open("scalerStandard.pkl", "rb") as f:
    scalerStandard = pickle.load(f)
with open("scalerRobust.pkl", "rb") as f:
    scalerRobust = pickle.load(f)
with open("labelScaler.pkl", "rb") as f:
    labelScaler = pickle.load(f)

for i, storm in enumerate(test_storms):
    test_storms[i][COLS_TO_SCALE_LOG] = pre_processing.scaleLog(
        test_storms[i][COLS_TO_SCALE_LOG]
    )
    test_storms[i][COLS_TO_SCALE_STANDARD] = scalerStandard.transform(
        test_storms[i][COLS_TO_SCALE_STANDARD]
    )
    test_storms[i][COLS_TO_SCALE_ROBUST] = scalerRobust.transform(
        test_storms[i][COLS_TO_SCALE_ROBUST]
    )

    for station in constants.STATIONS:
        test_storms[i][f"LDi_{station}"] = labelScaler.transform(
            test_storms[i][f"LDi_{station}"].values.reshape(-1, 1)
        )
        test_storms[i][f"Latitude_{station}"] = normalize_latitude(
            test_storms[i][f"Latitude_{station}"]
        )
        test_storms[i][f"Longitude_{station}"] = normalize_longitude(
            test_storms[i][f"Longitude_{station}"]
        )
        mlts = test_storms[i].apply(
            lambda row: encode_MLT(row[f"MLT_{station}"]), axis=1
        )
        test_storms[i][[f"MLT_{station}_sin", f"MLT_{station}_cos"]] = pd.DataFrame(
            mlts.tolist(), index=test_storms[i].index
        )
    test_storms[i] = test_storms[i].astype("float32")

/mnt/data/ldi-forecast-1and2-hours/ldi-torch/lib/python3.12/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/mnt/data/ldi-forecast-1and2-hours/ldi-torch/lib/python3.12/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator RobustScaler from version 1.6.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [14]:
def plot_storm_local(
    df_p, df_s, station, color, start_date, end_date, col_name, title_text="", ax=None
):
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(16, 8))

    sd = pd.to_datetime(start_date)
    ed = pd.to_datetime(end_date)

    df_pred = df_p[sd:ed].copy()
    df_sym = df_s[sd:ed].copy()

    month = df_pred.index[len(df_pred) // 2].month_name()
    year = df_pred.index[len(df_pred) // 2].year

    (lsym,) = ax.plot(
        df_sym[df_pred.index[0] : df_pred.index[-1]].index,
        df_sym[df_pred.index[0] : df_pred.index[-1]]["SYM_H"],
        color="black",
        alpha=0.4,
        label=f"SYM-H",
    )

    ax.set_xlabel(f"{month} of {year}", fontsize=25)
    ax.set_ylabel("Disturbance (nT)", fontsize=25)

    ax.set_title(title_text, fontsize=25)

    (lpred,) = ax.plot(
        df_pred.index,
        df_pred[f"Predicted_{col_name}"],
        label=f"Predicted LDi {station}",
        color=color,
        alpha=0.6,
        linewidth=2,
    )
    (lobs,) = ax.plot(
        df_pred.index,
        df_pred[f"Observed_{col_name}"],
        label=f"Observed LDi {station}",
        color="blue",
        alpha=0.6,
        linewidth=2,
    )
    (lconf,) = ax.plot(
        df_pred.index[0],
        df_pred[f"Observed_{col_name}"].iloc[0],
        label=f"90% confidence",
        color="lightblue",
        alpha=0.8,
        linewidth=2,
    )

    plt.setp(ax.spines.values(), lw=2, color="black", alpha=1)

    ax.xaxis_date()

    ax.fill_between(
        df_pred.index,
        df_pred[f"Quantile_05_{col_name}"],
        df_pred[f"Quantile_95_{col_name}"],
        alpha=0.8,
        interpolate=True,
        color="lightblue",
    )

    num_days = (df_pred.index[-1] - df_pred.index[0]).days
    if num_days < 14:
        interval = 1
    elif num_days <= 30:
        interval = 2
    else:
        interval = 3

    if num_days < 5:
        ax.xaxis.set_major_locator(mdates.HourLocator(interval=6))
    else:
        ax.xaxis.set_major_locator(mdates.DayLocator(interval=interval))
        ax.xaxis.set_minor_locator(mdates.HourLocator(interval=6))

    if num_days > 5:
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d"))
    else:
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %H:%M"))

    # Adjust the tick parameters for better visibility
    ax.tick_params(axis="both", which="major", labelsize=20, width=2, length=10)

    xticks = ax.get_xticklabels()
    if num_days < 5:
        for i in range(1, len(xticks), 2):
            xticks[i].set_visible(False)

    # Adjust the tick parameters for better visibility

    # Ensure that the grid is enabled and properly configured
    ax.grid(True, which="major", axis="both", linestyle="--", linewidth=0.5)

    # Ensure that the grid is enabled and properly configured
    ax.grid(True, which="major", axis="both", linestyle="--", linewidth=0.5)

    ax.set_xlim(df_pred.index[0], df_pred.index[-1])

    lines = [lobs, lpred, lsym, lconf]
    labels = [l.get_label() for l in lines]

    leg = ax.legend(
        lines,
        labels,
        bbox_to_anchor=(0.5, 1.3),
        loc="upper center",
        ncol=4,
        fancybox=True,
        prop={"size": 20},
    )
    leg.get_frame().set_alpha(None)
    leg.get_frame().set_facecolor((0, 0, 0, 0))
    leg.get_lines()[-1].set_linewidth(12.0)
    leg.get_frame().set_edgecolor("black")
    plt.setp(ax.spines.values(), lw=1, color="black", alpha=1)
    return fig, ax

In [15]:
def plot_storm_local_persistence(
    df_p,
    df_s,
    station,
    color,
    start_date,
    end_date,
    col_name_pred,
    col_name_obs,
    hours,
    title_text="",
    ax=None,
):
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(16, 8))

    sd = pd.to_datetime(start_date)
    ed = pd.to_datetime(end_date)

    df_pred = df_p[sd:ed].copy()
    df_sym = df_s[sd:ed].copy()

    month = df_pred.index[len(df_pred) // 2].month_name()
    year = df_pred.index[len(df_pred) // 2].year

    (lsym,) = ax.plot(
        df_sym[df_pred.index[0] : df_pred.index[-1]].index,
        df_sym[df_pred.index[0] : df_pred.index[-1]]["SYM_H"],
        color="black",
        alpha=0.4,
        label=f"SYM-H",
    )

    ax.set_xlabel(f"{month} of {year}", fontsize=25)
    ax.set_ylabel("Disturbance (nT)", fontsize=25)

    ax.set_title(title_text, fontsize=25)

    (lpred,) = ax.plot(
        df_pred.index,
        df_pred[col_name_pred],
        label=f"Persistence LDi {station} {hours}h",
        color=color,
        alpha=0.6,
        linewidth=2,
    )
    (lobs,) = ax.plot(
        df_pred.index,
        df_pred[col_name_obs],
        label=f"Observed LDi {station}",
        color="blue",
        alpha=0.6,
        linewidth=2,
    )

    plt.setp(ax.spines.values(), lw=2, color="black", alpha=1)

    ax.xaxis_date()

    num_days = (df_pred.index[-1] - df_pred.index[0]).days
    if num_days < 14:
        interval = 1
    elif num_days <= 30:
        interval = 2
    else:
        interval = 3

    if num_days < 5:
        ax.xaxis.set_major_locator(mdates.HourLocator(interval=6))
    else:
        ax.xaxis.set_major_locator(mdates.DayLocator(interval=interval))
        ax.xaxis.set_minor_locator(mdates.HourLocator(interval=6))

    if num_days > 5:
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d"))
    else:
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %H:%M"))

    # Adjust the tick parameters for better visibility
    ax.tick_params(axis="both", which="major", labelsize=20, width=2, length=10)

    xticks = ax.get_xticklabels()
    if num_days < 5:
        for i in range(1, len(xticks), 2):
            xticks[i].set_visible(False)

    # Adjust the tick parameters for better visibility

    # Ensure that the grid is enabled and properly configured
    ax.grid(True, which="major", axis="both", linestyle="--", linewidth=0.5)

    # Ensure that the grid is enabled and properly configured
    ax.grid(True, which="major", axis="both", linestyle="--", linewidth=0.5)

    ax.set_xlim(df_pred.index[0], df_pred.index[-1])

    lines = [lobs, lpred, lsym]
    labels = [l.get_label() for l in lines]

    leg = ax.legend(
        lines,
        labels,
        bbox_to_anchor=(0.5, 1.3),
        loc="upper center",
        ncol=4,
        fancybox=True,
        prop={"size": 20},
    )
    leg.get_frame().set_alpha(None)
    leg.get_frame().set_facecolor((0, 0, 0, 0))
    leg.get_lines()[-1].set_linewidth(12.0)
    leg.get_frame().set_edgecolor("black")
    plt.setp(ax.spines.values(), lw=1, color="black", alpha=1)
    return fig, ax

In [16]:
PLOT_STUFF = True
metrics_test = {}
for station in constants.STATIONS:
    metrics_test[station] = {}

summary_columns = [
    "Station",
    "StormIndex",
    "BFE SYM-H",
    "DTW SYM-H",
    "RMSE SYM-H",
    "R2 SYM-H",
]

for metric_col in [
    "BFE",
    "Better_BFE",
    "DTW",
    "RMSE",
    "Better_RMSE",
    "R2",
    "inside 90%",
]:
    for forecast_step in range(FORECAST_STEPS):
        summary_columns.append(f"{metric_col} {forecast_step + 1}h")

return_columns = [
    "StormIndex",
    "BFE SYM-H",
    "RMSE SYM-H",
]

for metric_col in ["BFE", "RMSE"]:
    for forecast_step in range(FORECAST_STEPS):
        return_columns.append(f"{metric_col} {forecast_step + 1}h")

metrics_return = [
    "Station",
    "Global_BFE_SYM-H",
    "Global_RMSE_SYM-H",
    "Global_R2_SYM-H",
]

for forecast_step in range(FORECAST_STEPS):
    for metric_col in [
        "Global_RMSE_",
        "Global_BFE_",
        "Global_R2_",
        "Global_inside_90%_",
    ]:
        metrics_return.append(f"{metric_col}{forecast_step + 1}h")

for forecast_step in range(FORECAST_STEPS):
    for metric_col in ["RMSE_persistence-", "BFE_persistence-", "R2_persistence-"]:
        metrics_return.append(f"{metric_col}{forecast_step + 1}h")

for metric_col in ["Better BFE", "Better RMSE"]:
    for forecast_step in range(FORECAST_STEPS):
        metrics_return.append(f"{metric_col} {forecast_step + 1}h")


def test_on_station(station, color, model, same_color=True):

    total_columns = COLS_TO_USE.copy()
    solar_wind_columns = COLS_TO_USE.copy()
    ldi_columns = station_columns[station]
    decoder_columns_position = station_columns_decoder[station]
    decoder_columns_ldi_inference = station_first_ldi[station]
    label_names = station_to_labels[station]

    summary_columns = [
        "Station",
        "StormIndex",
        "BFE SYM-H",
        "DTW SYM-H",
        "RMSE SYM-H",
        "R2 SYM_H",
    ]

    print(f"Testing model {model.__class__.__name__}")

    for i in range(FORECAST_STEPS):
        summary_columns.append(f"BFE {i+1}h")
    for i in range(FORECAST_STEPS):
        summary_columns.append(f"DTW {i+1}h")
    for i in range(FORECAST_STEPS):
        summary_columns.append(f"RMSE {i+1}h")
    for i in range(FORECAST_STEPS):
        summary_columns.append(f"R2 {i+1}h")
    for i in range(FORECAST_STEPS):
        summary_columns.append(f"inside 90% {i+1}h")

    for col in label_names:
        metrics_test[station][f"BFE_{col}"] = []
        metrics_test[station][f"DTW_{col}"] = []
        metrics_test[station][f"RMSE_{col}"] = []
        metrics_test[station][f"R2_{col}"] = []
        metrics_test[station][f"inside_90%_{col}"] = []

    original_index = f"LDi_{station}"

    for station_column in station_columns[station]:
        total_columns.append(station_column)
        for shift_suffix, shift_value in zip(SHIFT_NAMES, SHIFT_VALUES):
            total_columns.append(f"{station_column}{shift_suffix}")

    print(f"Evaluating for on station {station}")

    metrics_test[station][f"BFE_SYM-H"] = []
    metrics_test[station][f"DTW_SYM-H"] = []
    metrics_test[station][f"RMSE_SYM-H"] = []
    metrics_test[station][f"R2_SYM-H"] = []

    prediction_dfs_all = []
    prediction_dfs = {}
    global_sym = []

    skipped_indices = []
    global_presistences = []
    valid_indices = []

    for index_strm, ts in enumerate(test_storms):

        start_date = pd.to_datetime(storm_dates.TEST_STORMS[index_strm][0])
        end_date = pd.to_datetime(storm_dates.TEST_STORMS[index_strm][1])
        storm_index = storm_dates.TEST_STORMS[index_strm][2]

        if storm_index in blacklist_storms[station]:
            print(
                f"Skipping blacklisted test storm {storm_index} for {station} from {start_date} - {end_date}"
            )
            continue

        test_storm = ts.copy()

        test_storm[f"LDi_{station}"] = test_storm[f"LDi_{station}"].interpolate(
            limit_area="inside"
        )

        start_date = start_date
        end_date = end_date
        start_index = test_storm[f"LDi_{station}"].first_valid_index()
        end_index = test_storm[f"LDi_{station}"].last_valid_index()
        test_storm = test_storm.loc[start_index:end_index]

        for station_column in station_columns[station]:
            for shift_suffix, shift_value in zip(SHIFT_NAMES, SHIFT_VALUES):
                test_storm[f"{station_column}{shift_suffix}"] = test_storm[
                    f"{station_column}"
                ].shift(shift_value)

        valid = True
        nas = test_storm[f"LDi_{station}"].isna().sum()

        if nas > 0:
            print(
                f'Skipping test storm {storm_index} for station {station} from {start_date} - {end_date} - NAs: {test_storm[f"LDi_{station}"].isna().sum()}'
            )
            skipped_indices.append(storm_index)
            valid = False

        if not valid:
            fig, ax = plt.subplots(figsize=(10, 3))

            ax.plot(
                test_storm[start_date:end_date].index[0],
                test_storm[start_date:end_date][f"LDi_{station}"].iloc[0],
                color="black",
                alpha=0.1,
                label=f"MLT_{station}",
            )
            ax.plot(
                test_storm[start_date:end_date].index,
                test_storm[start_date:end_date][f"LDi_{station}"],
                color=color,
                label=f"LDi_{station}",
                alpha=0.5,
            )
            ax.plot(
                test_storm[start_date:end_date].index,
                test_storm[start_date:end_date][f"SYM_H"],
                color="blue",
                label=f"SYM_H",
                alpha=0.5,
            )
            ax.fill_between(
                test_storm[start_date:end_date].index,
                test_storm[start_date:end_date][f"LDi_{station}"].min(),
                test_storm[start_date:end_date][f"LDi_{station}"].max(),
                where=test_storm[start_date:end_date][f"LDi_{station}"].isna(),
                color="magenta",
                alpha=0.1,
                hatch="X",
            )
            ax.grid(True)
            ax.legend()
            print(
                "Test storm",
                storm_index,
                "from",
                start_date,
                "to",
                end_date,
                "ldi missing values",
                test_storm[start_date:end_date][f"LDi_{station}"].isna().sum(),
            )
            # plt.show();
            plt.close()
            continue

        ln = len(test_storm)
        valid_indices.append(storm_index)
        batch_encoder_solar_wind = torch.zeros(
            (ln - LOOKBACK, LOOKBACK, len(solar_wind_columns)), dtype=torch.float32
        )
        batch_encoder_ldi = torch.zeros(
            (ln - LOOKBACK, LOOKBACK, len(ldi_columns)), dtype=torch.float32
        )
        batch_decoder_first_ldi = torch.zeros((ln - LOOKBACK, 1), dtype=torch.float32)
        batch_decoder_ldi_position = torch.zeros(
            (ln - LOOKBACK, FORECAST_STEPS, NUM_FEATS_POSITION), dtype=torch.float32
        )
        labels = torch.zeros((ln - LOOKBACK, len(label_names)), dtype=torch.float32)
        indices_pred = []
        indices_pred_val = []
        print(
            f"Test storm number {storm_index}, from {start_date} "
            f"until {end_date}, number of batches: {ln}"
        )

        labels = np.zeros((ln - LOOKBACK, FORECAST_STEPS), dtype=np.float32)

        print(
            f"Shapes of the input tensors: {batch_encoder_solar_wind.shape}, {batch_encoder_ldi.shape}, {batch_decoder_first_ldi.shape}, {batch_decoder_ldi_position.shape}, {labels.shape}"
        )

        for i in range(ln - LOOKBACK):
            batch_encoder_solar_wind[i] = torch.tensor(
                test_storm.iloc[i : i + LOOKBACK][solar_wind_columns].values,
                dtype=torch.float32,
            )
            batch_encoder_ldi[i] = torch.tensor(
                test_storm.iloc[i : i + LOOKBACK][ldi_columns].values,
                dtype=torch.float32,
            )
            batch_decoder_first_ldi[i] = torch.tensor(
                test_storm.iloc[i + LOOKBACK - 1][decoder_columns_ldi_inference],
                dtype=torch.float32,
            )
            batch_decoder_ldi_position[i] = torch.tensor(
                test_storm.iloc[i + LOOKBACK - 1][
                    decoder_columns_position
                ].values.reshape(FORECAST_STEPS, NUM_FEATS_POSITION),
                dtype=torch.float32,
            )

            labels[i] = torch.tensor(
                test_storm.iloc[i + LOOKBACK - 1][label_names].values,
                dtype=torch.float32,
            )

            indices_pred.append(test_storm.iloc[i + LOOKBACK - 1].name)
            indices_pred_val.append(test_storm.iloc[i + LOOKBACK - 1][original_index])

        predictions_q05 = []
        predictions_q95 = []
        predictions_mse = []
        batch_decoder_first_ldi = batch_decoder_first_ldi.unsqueeze(-1)

        with torch.no_grad():
            if len(batch_encoder_solar_wind) > BATCH_SIZE:
                for i in range(len(batch_encoder_solar_wind) // BATCH_SIZE + 1):
                    mse, q05, q95 = model.inference(
                        batch_encoder_solar_wind[
                            i * BATCH_SIZE : (i + 1) * BATCH_SIZE
                        ].to(device),
                        batch_encoder_ldi[i * BATCH_SIZE : (i + 1) * BATCH_SIZE].to(
                            device
                        ),
                        batch_decoder_first_ldi[
                            i * BATCH_SIZE : (i + 1) * BATCH_SIZE
                        ].to(device),
                        batch_decoder_ldi_position[
                            i * BATCH_SIZE : (i + 1) * BATCH_SIZE
                        ].to(device),
                    )

                    predictions_q05.append(q05)
                    predictions_q95.append(q95)
                    predictions_mse.append(mse)

                predictions_q05 = torch.cat(predictions_q05)
                predictions_q95 = torch.cat(predictions_q95)
                predictions_mse = torch.cat(predictions_mse)
            else:
                predictions_mse, predictions_q05, predictions_q95 = model.inference(
                    batch_encoder_solar_wind.to(device),
                    batch_encoder_ldi.to(device),
                    batch_decoder_first_ldi.to(device),
                    batch_decoder_ldi_position.to(device),
                )

        predictions_mse_cat = (
            predictions_mse.squeeze().cpu().numpy().reshape(-1, FORECAST_STEPS)
        )
        predictions_q05_cat = (
            predictions_q05.squeeze().cpu().numpy().reshape(-1, FORECAST_STEPS)
        )
        predictions_q95_cat = (
            predictions_q95.squeeze().cpu().numpy().reshape(-1, FORECAST_STEPS)
        )

        indices_pred_val = np.array(indices_pred_val)
        indices_pred_val = labelScaler.inverse_transform(
            indices_pred_val.reshape(-1, 1)
        )

        pred_df = pd.DataFrame(
            data=indices_pred_val, index=indices_pred, columns=[original_index]
        )
        pred_df["storm_index"] = storm_index
        pred_df = pred_df.join(all_omni)

        for i, col in enumerate(label_names):
            pred_df[f"Observed_{col}"] = labelScaler.inverse_transform(
                labels[:, i].reshape(-1, 1)
            )
            pred_df[f"Predicted_{col}"] = labelScaler.inverse_transform(
                predictions_mse_cat[:, i].reshape(-1, 1)
            )
            pred_df[f"Quantile_05_{col}"] = labelScaler.inverse_transform(
                predictions_q05_cat[:, i].reshape(-1, 1)
            )
            pred_df[f"Quantile_95_{col}"] = labelScaler.inverse_transform(
                predictions_q95_cat[:, i].reshape(-1, 1)
            )

            pred_df[f"inside_90%_{col}"] = pred_df.apply(
                inside_quantile,
                axis=1,
                label=f"Observed_{col}",
                q_lower=f"Quantile_05_{col}",
                q_upper=f"Quantile_95_{col}",
            )

        prediction_dfs_all.append(pred_df.copy())
        pred_df.to_csv(
            f"./predictions/{station}/predictions-{station}-storm-{storm_index}.csv",
            index=True,
        )
        title_str = f'Test storm {storm_index} from {start_date.strftime("%Y-%m-%d")} until {end_date.strftime("%Y-%m-%d")}'

        if PLOT_STUFF:
            fig, ax = plt.subplots(figsize=(10, 4))
            ax = pred_df[start_date:end_date][original_index].plot(
                ax=ax,
                grid=True,
                label=f"Observed {original_index}",
                color="blue",
                alpha=0.8,
            )

            ax = pred_df[start_date:end_date]["SYM_H"].plot(
                ax=ax,
                grid=True,
                label=f"SYM-H",
                color="black",
                alpha=0.4,
            )

        comparison_omni = pred_df[start_date:end_date][[original_index, "SYM_H"]].copy()
        global_sym.append(comparison_omni.copy())
        rmse_mse = metrics.rmsem(
            comparison_omni[original_index], comparison_omni["SYM_H"]
        )
        metrics_test[station][f"RMSE_SYM-H"].append(rmse_mse)
        dtw_mse = metrics.compute_dtw(
            comparison_omni[original_index], comparison_omni["SYM_H"]
        )
        metrics_test[station][f"DTW_SYM-H"].append(dtw_mse)
        # storm_rmses_mse.append(rmse_mse)
        r2_mse = metrics.r2m(comparison_omni[original_index], comparison_omni["SYM_H"])
        metrics_test[station][f"R2_SYM-H"].append(r2_mse)
        # storm_r2s_mse.append(r2_mse)
        bfe_mse = metrics.calculate_BFE(
            comparison_omni[original_index], comparison_omni["SYM_H"]
        )
        metrics_test[station][f"BFE_SYM-H"].append(bfe_mse)
        # storm_bfes_mse.append(bfe_mse)

        title_str += f"\nSYM-H Metrics -> RMSE: {rmse_mse:.3f} | R2: {r2_mse:.3f} | BFE: {bfe_mse:.3f}"

        print(
            f"Metrics for SYM_H:\n\tRMSE: {rmse_mse:.3f} | R2: {r2_mse:.3f} | BFE: {bfe_mse:.3f} "
        )

        for i, col in enumerate(label_names):

            timestep_df = pd.DataFrame(
                index=indices_pred,
                columns=[
                    f"Observed_{col}",
                    f"Predicted_{col}",
                    f"Quantile_05_{col}",
                    f"Quantile_95_{col}",
                ],
            )

            timestep_df[f"Observed_{col}"] = pred_df[f"Observed_{col}"]
            timestep_df[f"Predicted_{col}"] = pred_df[f"Predicted_{col}"]
            timestep_df[f"Quantile_05_{col}"] = pred_df[f"Quantile_05_{col}"]
            timestep_df[f"Quantile_95_{col}"] = pred_df[f"Quantile_95_{col}"]
            # print('For the first time step, before shifting')
            # display(timestep_df)
            timestep_df.index = timestep_df.index.shift(
                periods=SHIFT_FORWARD[i], freq="5min"
            )
            # print('For the first time step, after shifting')
            # display(timestep_df)
            timestep_df["storm_index"] = storm_index
            timestep_df = timestep_df.dropna()
            timestep_df = timestep_df[start_date:end_date]
            timestep_df[f"inside_90%_{col}"] = timestep_df.apply(
                inside_quantile,
                axis=1,
                label=f"Observed_{col}",
                q_lower=f"Quantile_05_{col}",
                q_upper=f"Quantile_95_{col}",
            )

            rmse_mse = metrics.rmsem(
                timestep_df[f"Observed_{col}"], timestep_df[f"Predicted_{col}"]
            )
            metrics_test[station][f"RMSE_{col}"].append(rmse_mse)
            dtw_mse = metrics.compute_dtw(
                timestep_df[f"Observed_{col}"], timestep_df[f"Predicted_{col}"]
            )
            metrics_test[station][f"DTW_{col}"].append(dtw_mse)
            # storm_rmses_mse.append(rmse_mse)
            r2_mse = metrics.r2m(
                timestep_df[f"Observed_{col}"], timestep_df[f"Predicted_{col}"]
            )
            metrics_test[station][f"R2_{col}"].append(r2_mse)
            # storm_r2s_mse.append(r2_mse)
            bfe_mse = metrics.calculate_BFE(
                timestep_df[f"Observed_{col}"], timestep_df[f"Predicted_{col}"]
            )
            metrics_test[station][f"BFE_{col}"].append(bfe_mse)
            # storm_bfes_mse.append(bfe_mse)

            pct_inside_90_val = np.mean(timestep_df[f"inside_90%_{col}"])
            metrics_test[station][f"inside_90%_{col}"].append(pct_inside_90_val)
            # pct_inside_90.append(pct_inside_90_val)

            print(
                f"Metrics for {col}:\n\tRMSE: {rmse_mse:.3f} | R2: {r2_mse:.3f} | BFE: {bfe_mse:.3f} | Inside 90%: {pct_inside_90_val:.3f}"
            )

            if PLOT_STUFF:
                ax = timestep_df[start_date:end_date][[f"Predicted_{col}"]].plot(
                    ax=ax,
                    grid=True,
                    label=[f"Predicted_{col}"],
                    color=[metrics.COLORS_MULTIPLE[i]],
                    alpha=0.6,
                )

                ax.fill_between(
                    timestep_df[start_date:end_date].index,
                    timestep_df[start_date:end_date][f"Quantile_05_{col}"],
                    timestep_df[start_date:end_date][f"Quantile_95_{col}"],
                    alpha=0.2,
                    interpolate=True,
                    color=metrics.COLORS_LIGHTER_MULTIPLE[i],
                )
                title_str += f"\n{col} Metrics -> RMSE: {rmse_mse:.3f} | R2: {r2_mse:.3f} | BFE: {bfe_mse:.3f} | Inside 90%: {pct_inside_90_val:.3f}"

            prediction_dfs[f"storm_{storm_index}-{col}"] = timestep_df.copy()

        tper = test_storm[[f"LDi_{station}"]].copy()
        tper[f"LDi_{station}"] = labelScaler.inverse_transform(
            tper[f"LDi_{station}"].values.reshape(-1, 1)
        )
        for shift_val, shift_name in zip(SHIFT_VALUES, SHIFT_NAMES):
            tper[f"persistence_LDi_{station}{shift_name}"] = tper[
                f"LDi_{station}"
            ].shift(shift_val)

        tper = tper.dropna()
        for shift_name in SHIFT_NAMES:
            rmse_persistence = metrics.rmsem(
                tper[f"LDi_{station}"], tper[f"persistence_LDi_{station}{shift_name}"]
            )
            r2_persistence = metrics.r2m(
                tper[f"LDi_{station}"], tper[f"persistence_LDi_{station}{shift_name}"]
            )
            bfe_persistence = metrics.calculate_BFE(
                tper[f"LDi_{station}"], tper[f"persistence_LDi_{station}{shift_name}"]
            )
            print(
                f"Metrics for persistence {shift_name}:\n\tRMSE: {rmse_persistence:.3f} | R2: {r2_persistence:.3f} | BFE: {bfe_persistence:.3f} "
            )
        global_presistences.append(tper.copy())

        if PLOT_STUFF:
            ax.xaxis.set_major_locator(mdates.DayLocator())
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
            ax.grid(True, which="both", axis="x", linestyle="--", linewidth=0.5)
            ax.set_title(title_str)
            leg = ax.legend(
                bbox_to_anchor=(0.5, 1.55),
                loc="upper center",
                ncol=3,
                fancybox=True,
                prop={"size": 10},
            )
            leg.get_frame().set_edgecolor("black")
            plt.setp(ax.spines.values(), lw=2, color="black", alpha=1)
            plt.close()

            if storm_index == 104:
                for col_index, col in enumerate(label_names):
                    sd_may = pd.to_datetime("20240510")
                    if station == "ABG":
                        ed_may = pd.to_datetime("20240513")
                    else:
                        ed_may = pd.to_datetime("20240514")
                    plot_title = f'Peak of May 2024 storm at {station} from {sd_may.strftime("%Y-%m-%d")} until {ed_may.strftime("%Y-%m-%d")} {col_index + 1}h forecast\nBFE: {metrics.calculate_BFE(prediction_dfs[f"storm_{storm_index}-{col}"][sd_may:ed_may][f"Observed_{col}"],prediction_dfs[f"storm_{storm_index}-{col}"][sd_may:ed_may][f"Predicted_{col}"]):.3f} | RMSE: {metrics.rmsem(prediction_dfs[f"storm_{storm_index}-{col}"][sd_may:ed_may][f"Observed_{col}"],prediction_dfs[f"storm_{storm_index}-{col}"][sd_may:ed_may][f"Predicted_{col}"]):.3f} | R2: {metrics.r2m(prediction_dfs[f"storm_{storm_index}-{col}"][sd_may:ed_may][f"Observed_{col}"],prediction_dfs[f"storm_{storm_index}-{col}"][sd_may:ed_may][f"Predicted_{col}"]):.3f} | PICP: {np.mean(prediction_dfs[f"storm_{storm_index}-{col}"][sd_may:ed_may][f"inside_90%_{col}"]) * 100:.0f}%'
                    fig, ax = plot_storm_local(
                        prediction_dfs[f"storm_{storm_index}-{col}"],
                        pred_df,
                        station,
                        color,
                        sd_may,
                        ed_may,
                        col,
                        title_text=plot_title,
                    )
                    plt.tight_layout()
                    plt.savefig(
                        f'./figs{"_colors" if same_color else ""}/{station}/forecasts/forecast-ldi-{station}-{col}-test-storm-{storm_index}-may-storm.png',
                        transparent=True,
                        bbox_inches="tight",
                    )
                    plt.close()

                    fig, (ax_q, ax_color_q) = metrics.plot_evaluation_bfe_quantile(
                        prediction_dfs[f"storm_{storm_index}-{col}"][sd_may:ed_may][
                            [f"Observed_{col}", f"Predicted_{col}", f"inside_90%_{col}"]
                        ],
                        title="",
                        full_title=f'Peak of May 2024 storm at {station} from {sd_may.strftime("%Y-%m-%d")} until {ed_may.strftime("%Y-%m-%d")} {col_index + 1}h forecast\nBFE: {metrics.calculate_BFE(prediction_dfs[f"storm_{storm_index}-{col}"][sd_may:ed_may][f"Observed_{col}"],prediction_dfs[f"storm_{storm_index}-{col}"][sd_may:ed_may][f"Predicted_{col}"]):.3f} | PICP: {np.mean(prediction_dfs[f"storm_{storm_index}-{col}"][sd_may:ed_may][f"inside_90%_{col}"])* 100:.0f}%',
                        # full_title=f'Test storm {storm_index} at station {station} | {col_index + 1} hours ahead forecast\nFrom {start_date.strftime("%Y-%m-%d")} to {end_date.strftime("%Y-%m-%d")}',
                        plot_sym_bars=False,
                        xlabel_title=f"LDi {station}",
                    )
                    plt.tight_layout()
                    plt.savefig(
                        f'./figs{"_colors" if same_color else ""}/{station}/bfe/bfe-ldi-{station}-{col}-test-storm-{storm_index}-may-storm.png',
                        transparent=True,
                        bbox_inches="tight",
                    )
                    plt.close()

                # Persistence for the paper
                for col_index, shift_name in enumerate(SHIFT_NAMES):
                    sd_may = pd.to_datetime("20240510")
                    if station == "ABG":
                        ed_may = pd.to_datetime("20240513")
                    else:
                        ed_may = pd.to_datetime("20240514")
                    plot_title = f'Peak of May 2024 storm at {station} from {sd_may.strftime("%Y-%m-%d")} until {ed_may.strftime("%Y-%m-%d")} {col_index + 1}h persistence\nBFE: {metrics.calculate_BFE(tper[sd_may:ed_may][f"LDi_{station}"],tper[sd_may:ed_may][f"persistence_LDi_{station}{shift_name}"]):.3f} | RMSE: {metrics.rmsem(tper[sd_may:ed_may][f"LDi_{station}"],tper[sd_may:ed_may][f"persistence_LDi_{station}{shift_name}"]):.3f} | R2: {metrics.r2m(tper[sd_may:ed_may][f"LDi_{station}"],tper[sd_may:ed_may][f"persistence_LDi_{station}{shift_name}"]):.3f}'
                    fig, ax = plot_storm_local_persistence(
                        tper,
                        all_omni,
                        station,
                        color,
                        sd_may,
                        ed_may,
                        col_name_obs=f"LDi_{station}",
                        col_name_pred=f"persistence_LDi_{station}{shift_name}",
                        hours=col_index + 1,
                        title_text=plot_title,
                    )
                    plt.tight_layout()
                    plt.savefig(
                        f'./figs{"_colors" if same_color else ""}/{station}/forecasts/persistence-ldi-{station}-{col_index + 1}h-test-storm-{storm_index}-may-storm.png',
                        transparent=True,
                        bbox_inches="tight",
                    )
                    plt.close()

            elif storm_index == 108:
                for col_index, col in enumerate(label_names):
                    sd_october = pd.to_datetime("2024-10-10 09:00")
                    ed_october = pd.to_datetime("2024-10-12 09:00")
                    plot_title = f'Peak of October 2024 storm at {station} from {sd_october.strftime("%Y-%m-%d")} until {ed_october.strftime("%Y-%m-%d")} {col_index + 1}h forecast\nBFE: {metrics.calculate_BFE(prediction_dfs[f"storm_{storm_index}-{col}"][sd_october:ed_october][f"Observed_{col}"],prediction_dfs[f"storm_{storm_index}-{col}"][sd_october:ed_october][f"Predicted_{col}"]):.3f} | RMSE: {metrics.rmsem(prediction_dfs[f"storm_{storm_index}-{col}"][sd_october:ed_october][f"Observed_{col}"],prediction_dfs[f"storm_{storm_index}-{col}"][sd_october:ed_october][f"Predicted_{col}"]):.3f} | R2: {metrics.r2m(prediction_dfs[f"storm_{storm_index}-{col}"][sd_october:ed_october][f"Observed_{col}"],prediction_dfs[f"storm_{storm_index}-{col}"][sd_october:ed_october][f"Predicted_{col}"]):.3f} | PICP: {np.mean(prediction_dfs[f"storm_{storm_index}-{col}"][sd_october:ed_october][f"inside_90%_{col}"])* 100:.0f}%'
                    fig, ax = plot_storm_local(
                        prediction_dfs[f"storm_{storm_index}-{col}"],
                        pred_df,
                        station,
                        color,
                        sd_october,
                        ed_october,
                        col,
                        title_text=plot_title,
                    )
                    plt.tight_layout()
                    plt.savefig(
                        f'./figs{"_colors" if same_color else ""}/{station}/forecasts/forecast-ldi-{station}-{col}-test-storm-{storm_index}-october-storm.png',
                        transparent=True,
                        bbox_inches="tight",
                    )
                    plt.close()
                    # plt.show();

                    fig, (ax_q, ax_color_q) = metrics.plot_evaluation_bfe_quantile(
                        prediction_dfs[f"storm_{storm_index}-{col}"][
                            sd_october:ed_october
                        ][[f"Observed_{col}", f"Predicted_{col}", f"inside_90%_{col}"]],
                        title="",
                        full_title=f'Peak of October 2024 storm at {station} from {sd_october.strftime("%Y-%m-%d")} until {ed_october.strftime("%Y-%m-%d")} {col_index + 1}h forecast\nBFE: {metrics.calculate_BFE(prediction_dfs[f"storm_{storm_index}-{col}"][sd_october:ed_october][f"Observed_{col}"],prediction_dfs[f"storm_{storm_index}-{col}"][sd_october:ed_october][f"Predicted_{col}"]):.3f} | PICP: {np.mean(prediction_dfs[f"storm_{storm_index}-{col}"][sd_october:ed_october][f"inside_90%_{col}"])* 100:.0f}%',
                        # full_title=f'Test storm {storm_index} at station {station} | {col_index + 1} hours ahead forecast\nFrom {start_date.strftime("%Y-%m-%d")} to {end_date.strftime("%Y-%m-%d")}',
                        plot_sym_bars=False,
                        xlabel_title=f"LDi {station}",
                    )
                    plt.tight_layout()
                    plt.savefig(
                        f'./figs{"_colors" if same_color else ""}/{station}/bfe/bfe-ldi-{station}-{col}-test-storm-{storm_index}-october-storm.png',
                        transparent=True,
                        bbox_inches="tight",
                    )
                    plt.close()

        if SAVE_FIGS:
            for col_index, col in enumerate(label_names):
                if storm_index == 104:
                    sd_may = pd.to_datetime("20240510")
                    ed_may = pd.to_datetime("20240515")
                    plot_title = f'Test storm {storm_index} at {station} from {start_date.strftime("%Y-%m-%d")} until {end_date.strftime("%Y-%m-%d")} | {col_index + 1}h ahead forecast\nBFE: {metrics_test[station][f"BFE_{col}"][-1]:.3f} | RMSE: {metrics_test[station][f"RMSE_{col}"][-1]:.3f} | R2: {metrics_test[station][f"R2_{col}"][-1]:.3f} | PICP: {metrics_test[station][f"inside_90%_{col}"][-1] * 100:.0f}%'
                    fig, ax = plot_storm_local(
                        prediction_dfs[f"storm_{storm_index}-{col}"],
                        pred_df,
                        station,
                        color,
                        sd_may,
                        ed_may,
                        col,
                        title_text=plot_title,
                    )
                    plt.tight_layout()
                    # plt.savefig(f'./figs/{station}/forecasts/forecast-ldi-{station}-{col}-test-storm-{storm_index}-may-storm.png', transparent = True, bbox_inches='tight')
                    plt.close()

                if storm_index == 108:
                    sd_may = pd.to_datetime("20241009")
                    ed_may = pd.to_datetime("20241013")
                    plot_title = f'Test storm {storm_index} at {station} from {start_date.strftime("%Y-%m-%d")} until {end_date.strftime("%Y-%m-%d")} | {col_index + 1}h ahead forecast\nBFE: {metrics_test[station][f"BFE_{col}"][-1]:.3f} | RMSE: {metrics_test[station][f"RMSE_{col}"][-1]:.3f} | R2: {metrics_test[station][f"R2_{col}"][-1]:.3f} | PICP: {metrics_test[station][f"inside_90%_{col}"][-1] * 100:.0f}%'
                    fig, ax = plot_storm_local(
                        prediction_dfs[f"storm_{storm_index}-{col}"],
                        pred_df,
                        station,
                        color,
                        sd_may,
                        ed_may,
                        col,
                        title_text=plot_title,
                    )
                    plt.tight_layout()
                    # plt.savefig(f'./figs/{station}/forecasts/forecast-ldi-{station}-{col}-test-storm-{storm_index}-may-storm.png', transparent = True, bbox_inches='tight')
                    plt.close()

                plot_title = f'Test storm {storm_index} at {station} from {start_date.strftime("%Y-%m-%d")} until {end_date.strftime("%Y-%m-%d")} | {col_index + 1}h forecast\nBFE: {metrics_test[station][f"BFE_{col}"][-1]:.3f} | RMSE: {metrics_test[station][f"RMSE_{col}"][-1]:.3f} | R2: {metrics_test[station][f"R2_{col}"][-1]:.3f} | PICP: {metrics_test[station][f"inside_90%_{col}"][-1] * 100:.0f}%'
                fig, ax = plot_storm_local(
                    prediction_dfs[f"storm_{storm_index}-{col}"],
                    pred_df,
                    station,
                    color,
                    start_date,
                    end_date,
                    col,
                    title_text=plot_title,
                )
                plt.tight_layout()
                plt.savefig(
                    f'./figs{"_colors" if same_color else ""}/{station}/forecasts/forecast-ldi-{station}-{col}-test-storm-{storm_index}.png',
                    transparent=True,
                    bbox_inches="tight",
                )
                plt.close()
                fig, (ax_q, ax_color_q) = metrics.plot_evaluation_bfe_quantile(
                    prediction_dfs[f"storm_{storm_index}-{col}"][
                        [f"Observed_{col}", f"Predicted_{col}", f"inside_90%_{col}"]
                    ],
                    title="",
                    full_title=f'Test storm {storm_index} at {station} from {start_date.strftime("%Y-%m-%d")} until {end_date.strftime("%Y-%m-%d")} {col_index + 1}h forecast\nBFE: {metrics_test[station][f"BFE_{col}"][-1]:.3f} | PICP: {metrics_test[station][f"inside_90%_{col}"][-1] * 100:.0f}%',
                    # full_title=f'Test storm {storm_index} at station {station} | {col_index + 1} hours ahead forecast\nFrom {start_date.strftime("%Y-%m-%d")} to {end_date.strftime("%Y-%m-%d")}',
                    plot_sym_bars=False,
                    xlabel_title=f"LDi {station}",
                )
                plt.tight_layout()
                plt.savefig(
                    f'./figs{"_colors" if same_color else ""}/{station}/bfe/bfe-ldi-{station}-{col}-test-storm-{storm_index}.png',
                    transparent=True,
                    bbox_inches="tight",
                )
                plt.close()

        print("------------------------------------------------")

    metrics_test[station]["Mean_BFE_SYM-H"] = np.mean(
        metrics_test[station]["BFE_SYM-H"]
    )
    metrics_test[station]["Mean_DTW_SYM-H"] = np.mean(
        metrics_test[station]["DTW_SYM-H"]
    )
    metrics_test[station]["Mean_RMSE_SYM-H"] = np.mean(
        metrics_test[station]["RMSE_SYM-H"]
    )
    metrics_test[station]["Mean_R2_SYM-H"] = np.mean(metrics_test[station]["R2_SYM-H"])

    for col in label_names:
        metrics_test[station][f"Mean_BFE_{col}"] = np.mean(
            metrics_test[station][f"BFE_{col}"]
        )
        metrics_test[station][f"Mean_DTW_{col}"] = np.mean(
            metrics_test[station][f"DTW_{col}"]
        )
        metrics_test[station][f"Mean_RMSE_{col}"] = np.mean(
            metrics_test[station][f"RMSE_{col}"]
        )
        metrics_test[station][f"Mean_R2_{col}"] = np.mean(
            metrics_test[station][f"R2_{col}"]
        )
        metrics_test[station][f"Mean_inside_90%_{col}"] = np.mean(
            metrics_test[station][f"inside_90%_{col}"]
        )

    global_sym = pd.concat(global_sym)
    rmse_global_sym = metrics.rmsem(global_sym[original_index], global_sym["SYM_H"])
    dtw_global_sym = metrics.compute_dtw(
        global_sym[original_index], global_sym["SYM_H"]
    )
    r2_global_sym = metrics.r2m(global_sym[original_index], global_sym["SYM_H"])
    bfe_global_sym = metrics.calculate_BFE(
        global_sym[original_index], global_sym["SYM_H"]
    )

    metrics_test[station][f"Global_BFE_SYM-H"] = bfe_global_sym
    metrics_test[station][f"Global_RMSE_SYM-H"] = rmse_global_sym
    metrics_test[station][f"Global_R2_SYM-H"] = r2_global_sym
    metrics_test[station][f"Global_DTW_SYM-H"] = dtw_global_sym

    global_dfs = []
    for col in label_names:
        global_df = None
        for start_date, end_date, storm_index in storm_dates.TEST_STORMS:
            if global_df is None:
                if f"storm_{storm_index}-{col}" in prediction_dfs:
                    global_df = prediction_dfs[f"storm_{storm_index}-{col}"].copy()
            else:
                if f"storm_{storm_index}-{col}" in prediction_dfs:
                    global_df = pd.concat(
                        [global_df, prediction_dfs[f"storm_{storm_index}-{col}"]]
                    )
        rmse_global = metrics.rmsem(
            global_df[f"Observed_{col}"], global_df[f"Predicted_{col}"]
        )
        dtw_global = metrics.compute_dtw(
            global_df[f"Observed_{col}"], global_df[f"Predicted_{col}"]
        )
        r2_global = metrics.r2m(
            global_df[f"Observed_{col}"], global_df[f"Predicted_{col}"]
        )
        bfe_global = metrics.calculate_BFE(
            global_df[f"Observed_{col}"], global_df[f"Predicted_{col}"]
        )
        pct_inside_90_global = np.mean(global_df[f"inside_90%_{col}"])

        metrics_test[station][f"Global_BFE_{col}"] = bfe_global
        metrics_test[station][f"Global_DTW_{col}"] = dtw_global
        metrics_test[station][f"Global_RMSE_{col}"] = rmse_global
        metrics_test[station][f"Global_R2_{col}"] = r2_global
        metrics_test[station][f"Global_inside_90%_{col}"] = pct_inside_90_global
        global_dfs.append(global_df.copy())

    summary_df = pd.DataFrame(columns=summary_columns)

    names_metrics_dict = []
    mean_names_metrics_dict = []
    global_names_metrics_dict = []

    names_metrics_dict.append("BFE_SYM-H")
    mean_names_metrics_dict.append("Mean_BFE_SYM-H")
    global_names_metrics_dict.append("Global_BFE_SYM-H")
    names_metrics_dict.append("DTW_SYM-H")
    mean_names_metrics_dict.append("Mean_DTW_SYM-H")
    global_names_metrics_dict.append("Global_DTW_SYM-H")
    names_metrics_dict.append("RMSE_SYM-H")
    mean_names_metrics_dict.append("Mean_RMSE_SYM-H")
    global_names_metrics_dict.append("Global_RMSE_SYM-H")
    names_metrics_dict.append("R2_SYM-H")
    mean_names_metrics_dict.append("Mean_R2_SYM-H")
    global_names_metrics_dict.append("Global_R2_SYM-H")

    for col in label_names:
        names_metrics_dict.append(f"BFE_{col}")
        mean_names_metrics_dict.append(f"Mean_BFE_{col}")
        global_names_metrics_dict.append(f"Global_BFE_{col}")

    for col in label_names:
        names_metrics_dict.append(f"DTW_{col}")
        mean_names_metrics_dict.append(f"Mean_DTW_{col}")
        global_names_metrics_dict.append(f"Global_DTW_{col}")

    for col in label_names:
        names_metrics_dict.append(f"RMSE_{col}")
        mean_names_metrics_dict.append(f"Mean_RMSE_{col}")
        global_names_metrics_dict.append(f"Global_RMSE_{col}")

    for col in label_names:
        names_metrics_dict.append(f"R2_{col}")
        mean_names_metrics_dict.append(f"Mean_R2_{col}")
        global_names_metrics_dict.append(f"Global_R2_{col}")

    for col in label_names:
        names_metrics_dict.append(f"inside_90%_{col}")
        mean_names_metrics_dict.append(f"Mean_inside_90%_{col}")
        global_names_metrics_dict.append(f"Global_inside_90%_{col}")

    mets_to_return = ["Global_BFE_SYM-H", "Global_RMSE_SYM-H", "Global_R2_SYM-H"]
    for col in label_names:
        mets_to_return.append(f"Global_RMSE_{col}")
        mets_to_return.append(f"Global_BFE_{col}")
        mets_to_return.append(f"Global_R2_{col}")
        mets_to_return.append(f"Global_inside_90%_{col}")

    for col in label_names:
        mets_to_return.append(f"RMSE_persistence_{col}")
        mets_to_return.append(f"BFE_persistence_{col}")
        mets_to_return.append(f"R2_persistence_{col}")

    storm_i = 0
    for index_storm in valid_indices:
        row_to_add = [station, index_storm]
        for name in names_metrics_dict:
            row_to_add.append(metrics_test[station][name][storm_i])
        summary_df.loc[len(summary_df)] = row_to_add
        storm_i += 1

    means_to_add = [station, "Mean"]
    for name in mean_names_metrics_dict:
        means_to_add.append(metrics_test[station][name])
    summary_df.loc[len(summary_df)] = means_to_add

    globals_to_add = [station, "Global"]
    for name in global_names_metrics_dict:
        globals_to_add.append(metrics_test[station][name])
    summary_df.loc[len(summary_df)] = globals_to_add

    print("Metrics for the MSE output")
    display(summary_df.round(3))

    print("Mean and global MSE output")
    display(summary_df.iloc[-2:].round(3))

    print(summary_df.set_index("StormIndex").iloc[-2:, :5].round(3).to_string())

    tpers = pd.concat(global_presistences)
    str_return_persistence = ""

    for col in label_names:
        rmse_persistence = metrics.rmsem(
            tpers[f"LDi_{station}"], tpers[f"persistence_{col}"]
        )
        metrics_test[station][f"RMSE_persistence_{col}"] = rmse_persistence
        r2_persistence = metrics.r2m(
            tpers[f"LDi_{station}"], tpers[f"persistence_{col}"]
        )
        metrics_test[station][f"R2_persistence_{col}"] = r2_persistence
        bfe_persistence = metrics.calculate_BFE(
            tpers[f"LDi_{station}"], tpers[f"persistence_{col}"]
        )
        metrics_test[station][f"BFE_persistence_{col}"] = bfe_persistence
        print(
            f"Metrics for persistence {shift_name}:\n\tRMSE: {rmse_persistence:.3f} | R2: {r2_persistence:.3f} | BFE: {bfe_persistence:.3f} "
        )
        str_return_persistence += f"Metrics for persistence {col}:\n\tRMSE: {rmse_persistence:.3f} | BFE: {bfe_persistence:.3f}\n"

    metrics.plot_evaluation_bfe(
        global_sym[[original_index, "SYM_H"]],
        title=f"Global BFE LDi {station} vs SYM-H nowcast",
        xlabel_title=f"LDi {station}",
    )
    if SAVE_FIGS:
        plt.tight_layout()
        plt.savefig(
            f'./figs{"_colors" if same_color else ""}/{station}/bfe/sym-h-ldi-comparison-ldi-{station}-{col}.png',
            transparent=True,
            bbox_inches="tight",
        )
    # plt.show();
    plt.close()

    for i, col in enumerate(label_names):
        fig, ax = metrics.plot_evaluation_bfe_quantile(
            global_dfs[i][[f"Observed_{col}", f"Predicted_{col}", f"inside_90%_{col}"]],
            title=f"",
            # full_title=f'Global BFE LDi {station} {i + 1} hours',
            full_title=f'Global BFE for LDi {station} {i + 1} hour{"s" if i + 1 > 1 else ""} ahead forecast\nBFE: {metrics_test[station][f"Global_BFE_{col}"]:.3f} | PICP: {metrics_test[station][f"Global_inside_90%_{col}"] * 100:.3f}%',
            xlabel_title=f"LDi {station} (nT)",
        )
        if SAVE_FIGS:
            plt.tight_layout()
            plt.savefig(
                f'./figs{"_colors" if same_color else ""}/{station}/bfe/bfe-ldi-{station}-{col}.png',
                transparent=True,
                bbox_inches="tight",
            )
            plt.close()
        # plt.show();
        plt.close()
        metrics.plot_comparison_bfe(
            global_dfs[i][[f"Observed_{col}", f"Predicted_{col}"]],
            global_sym[[original_index, "SYM_H"]],
            title=f"LDi forecast {i + 1}h",
            title_to_compare=f"SYM-H nowcast",
            xlabel_title=f"LDi {station}",
            station=station,
            is_global=True,
        )
        if SAVE_FIGS:
            plt.tight_layout()
            plt.savefig(
                f'./figs{"_colors" if same_color else ""}/{station}/bfe/bfe-comparison-sym-ldi-{station}-{col}.png',
                transparent=True,
                bbox_inches="tight",
            )
            plt.close()
        # plt.show();
        plt.close()

    display(summary_df)
    str_return = f"Information for station {station}\n{summary_df[return_columns].iloc[-2:]}\n{str_return_persistence}\n"

    extra_mets = []
    for i in range(FORECAST_STEPS):
        extra_mets.append(
            f"{len(summary_df[summary_df[f'BFE {i+1}h'] < summary_df['BFE SYM-H']])}/{len(summary_df)}"
        )
    for i in range(FORECAST_STEPS):
        extra_mets.append(
            f"{len(summary_df[summary_df[f'RMSE {i+1}h'] < summary_df['RMSE SYM-H']])}/{len(summary_df)}"
        )

    return (
        str_return,
        [station]
        + [metrics_test[station][metric_name] for metric_name in mets_to_return]
        + extra_mets,
        summary_df,
    )

In [ ]:
summary_df = pd.DataFrame(columns=metrics_return)

station = constants.STATIONS[0]

color = "magenta"
sum_dfs_stations = []

str_summary = ""

str_return, metrics_station, sum_df_station = test_on_station(
    station, color, model, same_color=True
)
str_summary += str_return
sum_dfs_stations.append(sum_df_station)
summary_df.loc[len(summary_df)] = metrics_station

color = constants.COLOR_STATIONS[station]
_, _, _ = test_on_station(station, color, model, same_color=False)

Testing model RELDi
Evaluating for on station ABG
Test storm number 89, from 2017-09-06 00:00:00 until 2017-09-15 00:00:00, number of batches: 3168
Shapes of the input tensors: torch.Size([3072, 96, 6]), torch.Size([3072, 96, 5]), torch.Size([3072, 1]), torch.Size([3072, 2, 4]), (3072, 2)
Metrics for SYM_H:
	RMSE: 26.694 | R2: 0.571 | BFE: 51.318 
Metrics for LDi_ABG-1h:
	RMSE: 11.743 | R2: 0.917 | BFE: 20.252 | Inside 90%: 0.891
Metrics for LDi_ABG-2h:
	RMSE: 19.057 | R2: 0.782 | BFE: 33.472 | Inside 90%: 0.760
Metrics for persistence -1h:
	RMSE: 15.108 | R2: 0.840 | BFE: 21.632 
Metrics for persistence -2h:
	RMSE: 23.173 | R2: 0.625 | BFE: 30.768 
------------------------------------------------
Test storm number 90, from 2017-09-26 00:00:00 until 2017-10-04 00:00:00, number of batches: 2880
Shapes of the input tensors: torch.Size([2784, 96, 6]), torch.Size([2784, 96, 5]), torch.Size([2784, 1]), torch.Size([2784, 2, 4]), (2784, 2)
Metrics for SYM_H:
	RMSE: 15.757 | R2: 0.450 | BFE: 2

,Station,StormIndex,BFE SYM-H,DTW SYM-H,RMSE SYM-H,R2 SYM_H,BFE 1h,BFE 2h,DTW 1h,DTW 2h,RMSE 1h,RMSE 2h,R2 1h,R2 2h,inside 90% 1h,inside 90% 2h
0,ABG,89,51.318,19.426,26.694,0.571,20.252001,33.472000,7.563,12.157,11.743,19.057,0.917,0.782,0.891,0.760
1,ABG,90,22.046,12.131,15.757,0.450,10.234000,15.431000,5.656,8.121,8.172,11.436,0.852,0.710,0.919,0.862
2,ABG,91,29.525,12.406,18.686,0.799,11.295000,20.230000,5.444,8.680,8.581,13.624,0.958,0.893,0.951,0.868
3,ABG,92,31.660,9.375,14.601,0.772,13.660000,26.247999,7.476,11.575,10.766,16.971,0.876,0.692,0.816,0.669
4,ABG,94,29.084,12.950,16.151,0.717,9.298000,19.287001,5.315,9.934,7.852,14.647,0.933,0.767,0.931,0.781
5,ABG,95,37.289,14.424,19.457,0.751,18.204000,22.768999,7.989,12.140,12.428,18.433,0.899,0.777,0.890,0.736
6,ABG,96,24.434,11.188,18.377,0.782,13.289000,19.834999,7.138,10.686,11.087,15.496,0.920,0.845,0.853,0.765
7,ABG,97,17.032,18.278,20.975,0.785,14.359000,20.669001,7.339,11.392,11.008,16.248,0.941,0.871,0.897,0.790
8,ABG,98,7.317,6.739,8.807,0.806,8.692000,13.429000,4.820,7.694,6.588,10.297,0.892,0.735,0.966,0.893
9,ABG,99,37.337,9.740,15.927,0.882,30.148001,26.389000,8.339,11.464,14.285,16.602,0.905,0.872,0.948,0.842


Mean and global MSE output


,Station,StormIndex,BFE SYM-H,DTW SYM-H,RMSE SYM-H,R2 SYM_H,BFE 1h,BFE 2h,DTW 1h,DTW 2h,RMSE 1h,RMSE 2h,R2 1h,R2 2h,inside 90% 1h,inside 90% 2h
18,ABG,Mean,29.87,13.773,18.581,0.738,15.744000,24.489,6.736,10.528,10.463,16.104,0.921,0.81,0.910,0.797
19,ABG,Global,44.53,14.085,19.621,0.830,25.634001,40.916,6.848,10.769,11.129,17.194,0.945,0.87,0.912,0.797


           Station  BFE SYM-H  DTW SYM-H  RMSE SYM-H  R2 SYM_H
StormIndex                                                    
Mean           ABG      29.87     13.773      18.581     0.738
Global         ABG      44.53     14.085      19.621     0.830
Metrics for persistence -2h:
	RMSE: 13.092 | R2: 0.916 | BFE: 36.519 
Metrics for persistence -2h:
	RMSE: 19.043 | R2: 0.822 | BFE: 52.786 
Diff BFE (SYM-H nowcast - LDi forecast 1h):
Total: 18.896
Diff BFE (SYM-H nowcast - LDi forecast 2h):
Total: 3.614


...

,Station,StormIndex,BFE SYM-H,DTW SYM-H,RMSE SYM-H,R2 SYM_H,BFE 1h,BFE 2h,DTW 1h,DTW 2h,RMSE 1h,RMSE 2h,R2 1h,R2 2h,inside 90% 1h,inside 90% 2h
0,ABG,89,51.318,19.426,26.694,0.571,20.252001,33.472000,7.563,12.157,11.743,19.057,0.917,0.782,0.891,0.760
1,ABG,90,22.046,12.131,15.757,0.450,10.234000,15.431000,5.656,8.121,8.172,11.436,0.852,0.710,0.919,0.862
2,ABG,91,29.525,12.406,18.686,0.799,11.295000,20.230000,5.444,8.680,8.581,13.624,0.958,0.893,0.951,0.868
3,ABG,92,31.660,9.375,14.601,0.772,13.660000,26.247999,7.476,11.575,10.766,16.971,0.876,0.692,0.816,0.669
4,ABG,94,29.084,12.950,16.151,0.717,9.298000,19.287001,5.315,9.934,7.852,14.647,0.933,0.767,0.931,0.781
5,ABG,95,37.289,14.424,19.457,0.751,18.204000,22.768999,7.989,12.140,12.428,18.433,0.899,0.777,0.890,0.736
6,ABG,96,24.434,11.188,18.377,0.782,13.289000,19.834999,7.138,10.686,11.087,15.496,0.920,0.845,0.853,0.765
7,ABG,97,17.032,18.278,20.975,0.785,14.359000,20.669001,7.339,11.392,11.008,16.248,0.941,0.871,0.897,0.790
8,ABG,98,7.317,6.739,8.807,0.806,8.692000,13.429000,4.820,7.694,6.588,10.297,0.892,0.735,0.966,0.893
9,ABG,99,37.337,9.740,15.927,0.882,30.148001,26.389000,8.339,11.464,14.285,16.602,0.905,0.872,0.948,0.842


Mean and global MSE output


,Station,StormIndex,BFE SYM-H,DTW SYM-H,RMSE SYM-H,R2 SYM_H,BFE 1h,BFE 2h,DTW 1h,DTW 2h,RMSE 1h,RMSE 2h,R2 1h,R2 2h,inside 90% 1h,inside 90% 2h
18,ABG,Mean,29.87,13.773,18.581,0.738,15.744000,24.489,6.736,10.528,10.463,16.104,0.921,0.81,0.910,0.797
19,ABG,Global,44.53,14.085,19.621,0.830,25.634001,40.916,6.848,10.769,11.129,17.194,0.945,0.87,0.912,0.797


           Station  BFE SYM-H  DTW SYM-H  RMSE SYM-H  R2 SYM_H
StormIndex                                                    
Mean           ABG      29.87     13.773      18.581     0.738
Global         ABG      44.53     14.085      19.621     0.830
Metrics for persistence -2h:
	RMSE: 13.092 | R2: 0.916 | BFE: 36.519 
Metrics for persistence -2h:
	RMSE: 19.043 | R2: 0.822 | BFE: 52.786 
Diff BFE (SYM-H nowcast - LDi forecast 1h):
Total: 18.896
Diff BFE (SYM-H nowcast - LDi forecast 2h):
Total: 3.614


,Station,StormIndex,BFE SYM-H,DTW SYM-H,RMSE SYM-H,R2 SYM_H,BFE 1h,BFE 2h,DTW 1h,DTW 2h,RMSE 1h,RMSE 2h,R2 1h,R2 2h,inside 90% 1h,inside 90% 2h
0,ABG,89,51.317759,19.426228,26.693588,0.571418,20.251646,33.472221,7.562903,12.156519,11.742769,19.056625,0.917061,0.781570,0.890860,0.760123
1,ABG,90,22.046410,12.131271,15.756771,0.449524,10.234109,15.431261,5.656320,8.120926,8.171618,11.435536,0.851946,0.710054,0.919306,0.862039
2,ABG,91,29.525274,12.406395,18.686430,0.798634,11.295156,20.230370,5.444482,8.679736,8.581061,13.624454,0.957537,0.892954,0.950976,0.868113
3,ABG,92,31.660090,9.374594,14.600516,0.771799,13.660373,26.247932,7.476053,11.575391,10.766204,16.970741,0.875919,0.691694,0.815618,0.669414
4,ABG,94,29.083728,12.950148,16.150822,0.716789,9.298187,19.286816,5.314987,9.933503,7.851702,14.647009,0.933066,0.767074,0.931020,0.780911
5,ABG,95,37.289321,14.423606,19.456602,0.751398,18.203932,22.769079,7.989240,12.140351,12.428012,18.433069,0.898568,0.776866,0.890089,0.735827
6,ABG,96,24.433559,11.187536,18.376741,0.781544,13.288501,19.834919,7.137598,10.685829,11.086800,15.495961,0.920487,0.844667,0.853452,0.764751
7,ABG,97,17.032236,18.277609,20.974766,0.785222,14.358891,20.668797,7.339070,11.392482,11.008047,16.248240,0.940841,0.871113,0.897030,0.789819
8,ABG,98,7.316731,6.738902,8.807268,0.806347,8.692373,13.428818,4.819615,7.693836,6.587850,10.296861,0.891650,0.735301,0.966161,0.892842
9,ABG,99,37.336920,9.740413,15.927495,0.881778,30.147572,26.388510,8.339153,11.463689,14.284958,16.601902,0.904904,0.871555,0.947551,0.841882


In [ ]:
station = constants.STATIONS[1]
color = "magenta"

str_return, metrics_station, sum_df_station = test_on_station(
    station, color, model, same_color=True
)
str_summary += str_return
sum_dfs_stations.append(sum_df_station)
summary_df.loc[len(summary_df)] = metrics_station

color = constants.COLOR_STATIONS[station]
_, _, _ = test_on_station(station, color, model, same_color=False)

Testing model RELDi
Evaluating for on station MMB
Test storm number 89, from 2017-09-06 00:00:00 until 2017-09-15 00:00:00, number of batches: 3168
Shapes of the input tensors: torch.Size([3072, 96, 6]), torch.Size([3072, 96, 5]), torch.Size([3072, 1]), torch.Size([3072, 2, 4]), (3072, 2)
Metrics for SYM_H:
	RMSE: 15.483 | R2: 0.759 | BFE: 28.970 
Metrics for LDi_MMB-1h:
	RMSE: 13.880 | R2: 0.807 | BFE: 25.760 | Inside 90%: 0.941
Metrics for LDi_MMB-2h:
	RMSE: 22.257 | R2: 0.503 | BFE: 49.103 | Inside 90%: 0.858
Metrics for persistence -1h:
	RMSE: 15.056 | R2: 0.739 | BFE: 24.768 
Metrics for persistence -2h:
	RMSE: 20.894 | R2: 0.497 | BFE: 32.174 
------------------------------------------------
Test storm number 90, from 2017-09-26 00:00:00 until 2017-10-04 00:00:00, number of batches: 2880
Shapes of the input tensors: torch.Size([2784, 96, 6]), torch.Size([2784, 96, 5]), torch.Size([2784, 1]), torch.Size([2784, 2, 4]), (2784, 2)
Metrics for SYM_H:
	RMSE: 15.468 | R2: 0.427 | BFE: 2

,Station,StormIndex,BFE SYM-H,DTW SYM-H,RMSE SYM-H,R2 SYM_H,BFE 1h,BFE 2h,DTW 1h,DTW 2h,RMSE 1h,RMSE 2h,R2 1h,R2 2h,inside 90% 1h,inside 90% 2h
0,MMB,89,28.970,10.558,15.483,0.759,25.760000,49.103001,7.834,12.457,13.880,22.257,0.807,0.503,0.941,0.858
1,MMB,90,20.572,11.795,15.468,0.427,16.339001,22.264000,7.175,9.793,10.774,14.437,0.722,0.501,0.933,0.901
2,MMB,91,20.044,15.881,19.515,0.653,17.976999,18.676001,6.474,9.295,10.348,12.207,0.903,0.864,0.948,0.884
3,MMB,92,20.394,13.135,16.054,0.638,18.179001,25.665001,6.903,10.619,10.835,15.445,0.835,0.665,0.892,0.812
4,MMB,95,25.188,18.132,24.232,0.286,18.181000,19.059999,8.536,11.126,12.854,16.305,0.799,0.677,0.916,0.836
5,MMB,96,20.727,12.164,16.946,0.706,11.816000,14.473000,6.774,9.770,10.050,13.196,0.896,0.822,0.921,0.828
6,MMB,97,26.895,22.179,26.474,0.563,18.354000,33.808998,8.462,12.895,12.357,18.908,0.905,0.777,0.892,0.772
7,MMB,98,22.471,13.493,18.004,0.190,9.325000,12.853000,5.502,8.637,7.809,11.453,0.848,0.672,0.939,0.866
8,MMB,99,25.406,21.195,23.675,0.462,21.173000,26.246000,7.235,9.988,12.232,14.893,0.856,0.787,0.963,0.941
9,MMB,100,24.676,21.315,23.778,-0.276,17.882000,39.702999,6.168,10.066,9.708,15.650,0.787,0.447,0.959,0.861


Mean and global MSE output


,Station,StormIndex,BFE SYM-H,DTW SYM-H,RMSE SYM-H,R2 SYM_H,BFE 1h,BFE 2h,DTW 1h,DTW 2h,RMSE 1h,RMSE 2h,R2 1h,R2 2h,inside 90% 1h,inside 90% 2h
19,MMB,Mean,25.251,16.598,20.799,0.350,19.462000,27.850,7.287,10.779,11.534,16.166,0.838,0.675,0.926,0.842
20,MMB,Global,48.868,16.944,21.841,0.648,51.676998,77.945,7.472,11.048,12.723,17.774,0.881,0.767,0.927,0.842


           Station  BFE SYM-H  DTW SYM-H  RMSE SYM-H  R2 SYM_H
StormIndex                                                    
Mean           MMB     25.251     16.598      20.799     0.350
Global         MMB     48.868     16.944      21.841     0.648
Metrics for persistence -2h:
	RMSE: 13.976 | R2: 0.840 | BFE: 63.204 
Metrics for persistence -2h:
	RMSE: 18.071 | R2: 0.732 | BFE: 73.889 
Diff BFE (SYM-H nowcast - LDi forecast 1h):
Total: -2.809
Diff BFE (SYM-H nowcast - LDi forecast 2h):
Total: -29.077


...

,Station,StormIndex,BFE SYM-H,DTW SYM-H,RMSE SYM-H,R2 SYM_H,BFE 1h,BFE 2h,DTW 1h,DTW 2h,RMSE 1h,RMSE 2h,R2 1h,R2 2h,inside 90% 1h,inside 90% 2h
0,MMB,89,28.970,10.558,15.483,0.759,25.760000,49.103001,7.834,12.457,13.880,22.257,0.807,0.503,0.941,0.858
1,MMB,90,20.572,11.795,15.468,0.427,16.339001,22.264000,7.175,9.793,10.774,14.437,0.722,0.501,0.933,0.901
2,MMB,91,20.044,15.881,19.515,0.653,17.976999,18.676001,6.474,9.295,10.348,12.207,0.903,0.864,0.948,0.884
3,MMB,92,20.394,13.135,16.054,0.638,18.179001,25.665001,6.903,10.619,10.835,15.445,0.835,0.665,0.892,0.812
4,MMB,95,25.188,18.132,24.232,0.286,18.181000,19.059999,8.536,11.126,12.854,16.305,0.799,0.677,0.916,0.836
5,MMB,96,20.727,12.164,16.946,0.706,11.816000,14.473000,6.774,9.770,10.050,13.196,0.896,0.822,0.921,0.828
6,MMB,97,26.895,22.179,26.474,0.563,18.354000,33.808998,8.462,12.895,12.357,18.908,0.905,0.777,0.892,0.772
7,MMB,98,22.471,13.493,18.004,0.190,9.325000,12.853000,5.502,8.637,7.809,11.453,0.848,0.672,0.939,0.866
8,MMB,99,25.406,21.195,23.675,0.462,21.173000,26.246000,7.235,9.988,12.232,14.893,0.856,0.787,0.963,0.941
9,MMB,100,24.676,21.315,23.778,-0.276,17.882000,39.702999,6.168,10.066,9.708,15.650,0.787,0.447,0.959,0.861


Mean and global MSE output


,Station,StormIndex,BFE SYM-H,DTW SYM-H,RMSE SYM-H,R2 SYM_H,BFE 1h,BFE 2h,DTW 1h,DTW 2h,RMSE 1h,RMSE 2h,R2 1h,R2 2h,inside 90% 1h,inside 90% 2h
19,MMB,Mean,25.251,16.598,20.799,0.350,19.462000,27.850,7.287,10.779,11.534,16.166,0.838,0.675,0.926,0.842
20,MMB,Global,48.868,16.944,21.841,0.648,51.676998,77.945,7.472,11.048,12.723,17.774,0.881,0.767,0.927,0.842


           Station  BFE SYM-H  DTW SYM-H  RMSE SYM-H  R2 SYM_H
StormIndex                                                    
Mean           MMB     25.251     16.598      20.799     0.350
Global         MMB     48.868     16.944      21.841     0.648
Metrics for persistence -2h:
	RMSE: 13.976 | R2: 0.840 | BFE: 63.204 
Metrics for persistence -2h:
	RMSE: 18.071 | R2: 0.732 | BFE: 73.889 
Diff BFE (SYM-H nowcast - LDi forecast 1h):
Total: -2.809
Diff BFE (SYM-H nowcast - LDi forecast 2h):
Total: -29.077


,Station,StormIndex,BFE SYM-H,DTW SYM-H,RMSE SYM-H,R2 SYM_H,BFE 1h,BFE 2h,DTW 1h,DTW 2h,RMSE 1h,RMSE 2h,R2 1h,R2 2h,inside 90% 1h,inside 90% 2h
0,MMB,89,28.970003,10.557806,15.482639,0.759374,25.759645,49.102516,7.833791,12.456692,13.879745,22.256943,0.806618,0.502739,0.941381,0.858465
1,MMB,90,20.572237,11.794503,15.468181,0.427100,16.338943,22.263763,7.175279,9.792545,10.774351,14.437148,0.722040,0.500928,0.932755,0.901085
2,MMB,91,20.043994,15.881436,19.514948,0.653289,17.977362,18.675726,6.474243,9.294983,10.348242,12.206911,0.902509,0.864342,0.947505,0.884165
3,MMB,92,20.393878,13.135089,16.054302,0.637855,18.178793,25.665068,6.903223,10.619130,10.834508,15.445053,0.835063,0.664820,0.891540,0.812148
4,MMB,95,25.187811,18.132418,24.232353,0.285558,18.180920,19.060272,8.536189,11.126490,12.854163,16.304964,0.798969,0.676544,0.915542,0.836483
5,MMB,96,20.727015,12.164439,16.946199,0.705687,11.815985,14.473436,6.774405,9.770028,10.050018,13.195841,0.896486,0.821541,0.920555,0.827613
6,MMB,97,26.894582,22.179325,26.473621,0.562746,18.354452,33.809193,8.462420,12.894705,12.356864,18.908106,0.904737,0.776949,0.892017,0.771693
7,MMB,98,22.471309,13.492586,18.003632,0.190049,9.324956,12.853023,5.502089,8.636580,7.808781,11.452759,0.847628,0.672238,0.939262,0.865510
8,MMB,99,25.406242,21.194956,23.674667,0.461732,21.173141,26.246243,7.235442,9.987666,12.231531,14.893165,0.856321,0.786988,0.963363,0.940609
9,MMB,100,24.676463,21.314918,23.777777,-0.275570,17.881750,39.703365,6.167570,10.066218,9.707876,15.650310,0.787377,0.447404,0.958806,0.861143


In [ ]:
station = constants.STATIONS[2]
color = "magenta"

str_return, metrics_station, sum_df_station = test_on_station(
    station, color, model, same_color=True
)
str_summary += str_return
sum_dfs_stations.append(sum_df_station)
summary_df.loc[len(summary_df)] = metrics_station

color = constants.COLOR_STATIONS[station]
_, _, _ = test_on_station(station, color, model, same_color=False)

Testing model RELDi
Evaluating for on station CLF
Test storm number 89, from 2017-09-06 00:00:00 until 2017-09-15 00:00:00, number of batches: 3168
Shapes of the input tensors: torch.Size([3072, 96, 6]), torch.Size([3072, 96, 5]), torch.Size([3072, 1]), torch.Size([3072, 2, 4]), (3072, 2)
Metrics for SYM_H:
	RMSE: 20.225 | R2: 0.429 | BFE: 29.023 
Metrics for LDi_CLF-1h:
	RMSE: 12.581 | R2: 0.779 | BFE: 19.815 | Inside 90%: 0.936
Metrics for LDi_CLF-2h:
	RMSE: 17.295 | R2: 0.582 | BFE: 31.102 | Inside 90%: 0.846
Metrics for persistence -1h:
	RMSE: 14.432 | R2: 0.674 | BFE: 18.977 
Metrics for persistence -2h:
	RMSE: 18.465 | R2: 0.466 | BFE: 23.340 
------------------------------------------------
Test storm number 90, from 2017-09-26 00:00:00 until 2017-10-04 00:00:00, number of batches: 2880
Shapes of the input tensors: torch.Size([2784, 96, 6]), torch.Size([2784, 96, 5]), torch.Size([2784, 1]), torch.Size([2784, 2, 4]), (2784, 2)
Metrics for SYM_H:
	RMSE: 20.098 | R2: -0.875 | BFE: 

,Station,StormIndex,BFE SYM-H,DTW SYM-H,RMSE SYM-H,R2 SYM_H,BFE 1h,BFE 2h,DTW 1h,DTW 2h,RMSE 1h,RMSE 2h,R2 1h,R2 2h,inside 90% 1h,inside 90% 2h
0,CLF,89,29.023,12.591,20.225,0.429,19.815001,31.101999,7.882,11.656,12.581,17.295,0.779,0.582,0.936,0.846
1,CLF,90,11.599,15.927,20.098,-0.875,10.888000,11.859000,7.056,9.527,9.776,12.311,0.556,0.296,0.917,0.877
2,CLF,91,51.787,21.559,30.035,-0.034,19.524000,27.134001,7.531,12.040,11.865,17.454,0.839,0.651,0.945,0.837
3,CLF,92,32.090,12.921,16.916,0.359,26.579000,33.106998,8.230,11.748,12.634,16.299,0.642,0.404,0.873,0.785
4,CLF,93,24.715,16.407,20.450,-0.138,13.680000,17.554001,6.421,9.360,9.131,12.003,0.773,0.608,0.969,0.938
5,CLF,94,19.697,26.323,28.229,-1.498,9.855000,17.781000,4.947,8.418,6.567,11.006,0.865,0.620,0.986,0.934
6,CLF,95,22.405,18.205,24.789,0.103,15.226000,21.454000,8.732,13.367,13.010,19.156,0.753,0.464,0.939,0.831
7,CLF,96,29.375,14.182,22.895,0.054,15.139000,18.035000,6.860,9.638,10.907,13.363,0.785,0.678,0.897,0.856
8,CLF,97,54.820,27.407,35.520,-0.632,27.681999,30.516001,8.731,12.350,12.929,16.783,0.784,0.636,0.891,0.754
9,CLF,98,17.530,10.245,16.614,-0.279,15.617000,25.833000,6.371,9.190,9.305,13.143,0.599,0.199,0.956,0.854


Mean and global MSE output


,Station,StormIndex,BFE SYM-H,DTW SYM-H,RMSE SYM-H,R2 SYM_H,BFE 1h,BFE 2h,DTW 1h,DTW 2h,RMSE 1h,RMSE 2h,R2 1h,R2 2h,inside 90% 1h,inside 90% 2h
21,CLF,Mean,32.106,19.112,25.318,-0.236,20.372000,26.983000,7.267,10.716,11.382,15.715,0.767,0.555,0.933,0.858
22,CLF,Global,73.670,19.979,28.088,0.064,64.436996,74.647003,7.460,10.966,12.653,17.308,0.810,0.645,0.933,0.859


           Station  BFE SYM-H  DTW SYM-H  RMSE SYM-H  R2 SYM_H
StormIndex                                                    
Mean           CLF     32.106     19.112      25.318    -0.236
Global         CLF     73.670     19.979      28.088     0.064
Metrics for persistence -2h:
	RMSE: 13.115 | R2: 0.774 | BFE: 66.489 
Metrics for persistence -2h:
	RMSE: 17.000 | R2: 0.620 | BFE: 100.731 
Diff BFE (SYM-H nowcast - LDi forecast 1h):
Total: 9.233
Diff BFE (SYM-H nowcast - LDi forecast 2h):
Total: -0.977


...

,Station,StormIndex,BFE SYM-H,DTW SYM-H,RMSE SYM-H,R2 SYM_H,BFE 1h,BFE 2h,DTW 1h,DTW 2h,RMSE 1h,RMSE 2h,R2 1h,R2 2h,inside 90% 1h,inside 90% 2h
0,CLF,89,29.023,12.591,20.225,0.429,19.815001,31.101999,7.882,11.656,12.581,17.295,0.779,0.582,0.936,0.846
1,CLF,90,11.599,15.927,20.098,-0.875,10.888000,11.859000,7.056,9.527,9.776,12.311,0.556,0.296,0.917,0.877
2,CLF,91,51.787,21.559,30.035,-0.034,19.524000,27.134001,7.531,12.040,11.865,17.454,0.839,0.651,0.945,0.837
3,CLF,92,32.090,12.921,16.916,0.359,26.579000,33.106998,8.230,11.748,12.634,16.299,0.642,0.404,0.873,0.785
4,CLF,93,24.715,16.407,20.450,-0.138,13.680000,17.554001,6.421,9.360,9.131,12.003,0.773,0.608,0.969,0.938
5,CLF,94,19.697,26.323,28.229,-1.498,9.855000,17.781000,4.947,8.418,6.567,11.006,0.865,0.620,0.986,0.934
6,CLF,95,22.405,18.205,24.789,0.103,15.226000,21.454000,8.732,13.367,13.010,19.156,0.753,0.464,0.939,0.831
7,CLF,96,29.375,14.182,22.895,0.054,15.139000,18.035000,6.860,9.638,10.907,13.363,0.785,0.678,0.897,0.856
8,CLF,97,54.820,27.407,35.520,-0.632,27.681999,30.516001,8.731,12.350,12.929,16.783,0.784,0.636,0.891,0.754
9,CLF,98,17.530,10.245,16.614,-0.279,15.617000,25.833000,6.371,9.190,9.305,13.143,0.599,0.199,0.956,0.854


Mean and global MSE output


,Station,StormIndex,BFE SYM-H,DTW SYM-H,RMSE SYM-H,R2 SYM_H,BFE 1h,BFE 2h,DTW 1h,DTW 2h,RMSE 1h,RMSE 2h,R2 1h,R2 2h,inside 90% 1h,inside 90% 2h
21,CLF,Mean,32.106,19.112,25.318,-0.236,20.372000,26.983000,7.267,10.716,11.382,15.715,0.767,0.555,0.933,0.858
22,CLF,Global,73.670,19.979,28.088,0.064,64.436996,74.647003,7.460,10.966,12.653,17.308,0.810,0.645,0.933,0.859


           Station  BFE SYM-H  DTW SYM-H  RMSE SYM-H  R2 SYM_H
StormIndex                                                    
Mean           CLF     32.106     19.112      25.318    -0.236
Global         CLF     73.670     19.979      28.088     0.064
Metrics for persistence -2h:
	RMSE: 13.115 | R2: 0.774 | BFE: 66.489 
Metrics for persistence -2h:
	RMSE: 17.000 | R2: 0.620 | BFE: 100.731 
Diff BFE (SYM-H nowcast - LDi forecast 1h):
Total: 9.233
Diff BFE (SYM-H nowcast - LDi forecast 2h):
Total: -0.977


,Station,StormIndex,BFE SYM-H,DTW SYM-H,RMSE SYM-H,R2 SYM_H,BFE 1h,BFE 2h,DTW 1h,DTW 2h,RMSE 1h,RMSE 2h,R2 1h,R2 2h,inside 90% 1h,inside 90% 2h
0,CLF,89,29.022701,12.591280,20.225471,0.428736,19.815281,31.102060,7.882171,11.656203,12.581303,17.294760,0.778950,0.582296,0.935596,0.845739
1,CLF,90,11.598647,15.927202,20.098200,-0.875462,10.888056,11.859143,7.056229,9.526742,9.775786,12.310779,0.556292,0.296337,0.917137,0.876790
2,CLF,91,51.786812,21.558681,30.034981,-0.034254,19.523720,27.133652,7.531140,12.039882,11.864594,17.453571,0.838609,0.650746,0.945336,0.837310
3,CLF,92,32.090188,12.920742,16.915661,0.358527,26.579252,33.106655,8.229828,11.748114,12.633847,16.299416,0.642174,0.404414,0.872885,0.785249
4,CLF,93,24.715146,16.406712,20.449997,-0.137736,13.679865,17.554382,6.421145,9.359586,9.131367,12.003187,0.773156,0.608034,0.969197,0.937527
5,CLF,94,19.697226,26.322950,28.228559,-1.498036,9.855026,17.781281,4.947400,8.418427,6.567156,11.005636,0.864800,0.620290,0.986117,0.933623
6,CLF,95,22.404838,18.205199,24.789005,0.102701,15.226352,21.453890,8.731922,13.367037,13.009989,19.156385,0.752843,0.464146,0.938681,0.831084
7,CLF,96,29.374508,14.181984,22.894989,0.053858,15.139376,18.035261,6.860385,9.637814,10.907060,13.362988,0.785271,0.677683,0.897030,0.856151
8,CLF,97,54.820490,27.406575,35.519718,-0.631544,27.681923,30.516241,8.730739,12.350303,12.928798,16.782640,0.783840,0.635766,0.890860,0.754339
9,CLF,98,17.530063,10.245236,16.614473,-0.279384,15.616583,25.833092,6.370857,9.190394,9.305398,13.142600,0.598674,0.199447,0.955748,0.853796


In [ ]:
station = constants.STATIONS[3]
color = "magenta"

str_return, metrics_station, sum_df_station = test_on_station(
    station, color, model, same_color=True
)
str_summary += str_return
sum_dfs_stations.append(sum_df_station)
summary_df.loc[len(summary_df)] = metrics_station

color = constants.COLOR_STATIONS[station]
_, _, _ = test_on_station(station, color, model, same_color=False)

Testing model RELDi
Evaluating for on station TUC
Test storm number 89, from 2017-09-06 00:00:00 until 2017-09-15 00:00:00, number of batches: 3168
Shapes of the input tensors: torch.Size([3072, 96, 6]), torch.Size([3072, 96, 5]), torch.Size([3072, 1]), torch.Size([3072, 2, 4]), (3072, 2)
Metrics for SYM_H:
	RMSE: 14.876 | R2: 0.785 | BFE: 27.377 
Metrics for LDi_TUC-1h:
	RMSE: 14.237 | R2: 0.803 | BFE: 24.439 | Inside 90%: 0.914
Metrics for LDi_TUC-2h:
	RMSE: 21.635 | R2: 0.545 | BFE: 40.388 | Inside 90%: 0.804
Metrics for persistence -1h:
	RMSE: 16.256 | R2: 0.710 | BFE: 29.966 
Metrics for persistence -2h:
	RMSE: 21.567 | R2: 0.490 | BFE: 35.063 
------------------------------------------------
Test storm number 90, from 2017-09-26 00:00:00 until 2017-10-04 00:00:00, number of batches: 2880
Shapes of the input tensors: torch.Size([2784, 96, 6]), torch.Size([2784, 96, 5]), torch.Size([2784, 1]), torch.Size([2784, 2, 4]), (2784, 2)
Metrics for SYM_H:
	RMSE: 15.174 | R2: 0.343 | BFE: 1

,Station,StormIndex,BFE SYM-H,DTW SYM-H,RMSE SYM-H,R2 SYM_H,BFE 1h,BFE 2h,DTW 1h,DTW 2h,RMSE 1h,RMSE 2h,R2 1h,R2 2h,inside 90% 1h,inside 90% 2h
0,TUC,89,27.377,9.472,14.876,0.785,24.438999,40.388000,8.773,13.715,14.237,21.635,0.803,0.545,0.914,0.804
1,TUC,90,18.354,12.391,15.174,0.343,14.391000,12.627000,7.339,9.964,11.046,13.499,0.652,0.480,0.908,0.837
2,TUC,91,26.054,16.061,19.670,0.636,9.840000,15.406000,6.211,9.649,8.990,13.431,0.924,0.830,0.953,0.843
3,TUC,92,40.312,12.424,18.021,0.200,33.561001,57.998001,7.436,11.480,11.850,17.892,0.654,0.211,0.886,0.782
4,TUC,93,13.520,16.684,18.357,0.221,9.607000,11.927000,5.590,8.407,8.082,11.723,0.849,0.682,0.974,0.953
5,TUC,94,32.468,25.959,29.452,-1.617,7.582000,10.704000,5.656,9.229,7.599,11.869,0.826,0.575,0.938,0.784
6,TUC,95,20.286,17.243,23.831,0.299,15.744000,18.662001,8.874,12.765,13.464,18.322,0.776,0.586,0.895,0.788
7,TUC,96,24.614,11.846,16.795,0.808,19.323999,24.774000,7.131,10.703,12.201,15.508,0.899,0.836,0.909,0.796
8,TUC,97,39.197,27.122,30.900,0.164,19.197001,24.764999,8.871,12.645,12.853,18.149,0.855,0.712,0.882,0.763
9,TUC,98,16.930,9.662,12.404,0.572,19.230000,20.191000,6.575,10.029,10.551,14.185,0.690,0.440,0.917,0.818


Mean and global MSE output


,Station,StormIndex,BFE SYM-H,DTW SYM-H,RMSE SYM-H,R2 SYM_H,BFE 1h,BFE 2h,DTW 1h,DTW 2h,RMSE 1h,RMSE 2h,R2 1h,R2 2h,inside 90% 1h,inside 90% 2h
21,TUC,Mean,24.772,15.757,20.244,0.322,18.754000,24.930000,7.267,10.777,11.671,16.069,0.810,0.630,0.921,0.826
22,TUC,Global,38.910,16.037,21.552,0.617,37.391998,37.348999,7.388,10.894,12.727,16.995,0.867,0.762,0.922,0.831


           Station  BFE SYM-H  DTW SYM-H  RMSE SYM-H  R2 SYM_H
StormIndex                                                    
Mean           TUC     24.772     15.757      20.244     0.322
Global         TUC     38.910     16.037      21.552     0.617
Metrics for persistence -2h:
	RMSE: 13.454 | R2: 0.834 | BFE: 46.376 
Metrics for persistence -2h:
	RMSE: 17.293 | R2: 0.727 | BFE: 54.432 
Diff BFE (SYM-H nowcast - LDi forecast 1h):
Total: 1.519
Diff BFE (SYM-H nowcast - LDi forecast 2h):
Total: 1.561


...

,Station,StormIndex,BFE SYM-H,DTW SYM-H,RMSE SYM-H,R2 SYM_H,BFE 1h,BFE 2h,DTW 1h,DTW 2h,RMSE 1h,RMSE 2h,R2 1h,R2 2h,inside 90% 1h,inside 90% 2h
0,TUC,89,27.377,9.472,14.876,0.785,24.438999,40.388000,8.773,13.715,14.237,21.635,0.803,0.545,0.914,0.804
1,TUC,90,18.354,12.391,15.174,0.343,14.391000,12.627000,7.339,9.964,11.046,13.499,0.652,0.480,0.908,0.837
2,TUC,91,26.054,16.061,19.670,0.636,9.840000,15.406000,6.211,9.649,8.990,13.431,0.924,0.830,0.953,0.843
3,TUC,92,40.312,12.424,18.021,0.200,33.561001,57.998001,7.436,11.480,11.850,17.892,0.654,0.211,0.886,0.782
4,TUC,93,13.520,16.684,18.357,0.221,9.607000,11.927000,5.590,8.407,8.082,11.723,0.849,0.682,0.974,0.953
5,TUC,94,32.468,25.959,29.452,-1.617,7.582000,10.704000,5.656,9.229,7.599,11.869,0.826,0.575,0.938,0.784
6,TUC,95,20.286,17.243,23.831,0.299,15.744000,18.662001,8.874,12.765,13.464,18.322,0.776,0.586,0.895,0.788
7,TUC,96,24.614,11.846,16.795,0.808,19.323999,24.774000,7.131,10.703,12.201,15.508,0.899,0.836,0.909,0.796
8,TUC,97,39.197,27.122,30.900,0.164,19.197001,24.764999,8.871,12.645,12.853,18.149,0.855,0.712,0.882,0.763
9,TUC,98,16.930,9.662,12.404,0.572,19.230000,20.191000,6.575,10.029,10.551,14.185,0.690,0.440,0.917,0.818


Mean and global MSE output


,Station,StormIndex,BFE SYM-H,DTW SYM-H,RMSE SYM-H,R2 SYM_H,BFE 1h,BFE 2h,DTW 1h,DTW 2h,RMSE 1h,RMSE 2h,R2 1h,R2 2h,inside 90% 1h,inside 90% 2h
21,TUC,Mean,24.772,15.757,20.244,0.322,18.754000,24.930000,7.267,10.777,11.671,16.069,0.810,0.630,0.921,0.826
22,TUC,Global,38.910,16.037,21.552,0.617,37.391998,37.348999,7.388,10.894,12.727,16.995,0.867,0.762,0.922,0.831


           Station  BFE SYM-H  DTW SYM-H  RMSE SYM-H  R2 SYM_H
StormIndex                                                    
Mean           TUC     24.772     15.757      20.244     0.322
Global         TUC     38.910     16.037      21.552     0.617
Metrics for persistence -2h:
	RMSE: 13.454 | R2: 0.834 | BFE: 46.376 
Metrics for persistence -2h:
	RMSE: 17.293 | R2: 0.727 | BFE: 54.432 
Diff BFE (SYM-H nowcast - LDi forecast 1h):
Total: 1.519
Diff BFE (SYM-H nowcast - LDi forecast 2h):
Total: 1.561


,Station,StormIndex,BFE SYM-H,DTW SYM-H,RMSE SYM-H,R2 SYM_H,BFE 1h,BFE 2h,DTW 1h,DTW 2h,RMSE 1h,RMSE 2h,R2 1h,R2 2h,inside 90% 1h,inside 90% 2h
0,TUC,89,27.377050,9.471643,14.876406,0.785007,24.438717,40.387577,8.772961,13.715310,14.236505,21.634676,0.803104,0.545295,0.914385,0.804474
1,TUC,90,18.353788,12.390967,15.174352,0.342907,14.390879,12.626810,7.338869,9.963865,11.046460,13.498765,0.651781,0.480011,0.907592,0.837310
2,TUC,91,26.053659,16.061210,19.670437,0.635996,9.839642,15.405602,6.211276,9.648616,8.990067,13.430912,0.923967,0.830297,0.953145,0.843384
3,TUC,92,40.312241,12.424412,18.020643,0.199571,33.560577,57.998047,7.436033,11.480058,11.850436,17.891600,0.653861,0.210994,0.885900,0.782213
4,TUC,93,13.520003,16.683705,18.357494,0.221382,9.606622,11.927454,5.589860,8.407442,8.081972,11.723221,0.849085,0.682465,0.974403,0.952711
5,TUC,94,32.467527,25.958712,29.452412,-1.617054,7.581527,10.703839,5.655639,9.228788,7.598772,11.869062,0.825796,0.574985,0.937961,0.783948
6,TUC,95,20.286297,17.243247,23.830587,0.299382,15.743934,18.661596,8.874243,12.765076,13.463799,18.321762,0.776361,0.585861,0.895488,0.788276
7,TUC,96,24.613627,11.846109,16.795305,0.807783,19.324160,24.773848,7.131021,10.702872,12.201094,15.507909,0.898559,0.836121,0.909371,0.795989
8,TUC,97,39.197167,27.121824,30.899662,0.164186,19.197283,24.765493,8.871183,12.644924,12.853127,18.148508,0.855383,0.711674,0.881990,0.762823
9,TUC,98,16.929598,9.661774,12.404138,0.571647,19.230019,20.190666,6.574748,10.029345,10.550587,14.184690,0.690099,0.439844,0.916703,0.817787


In [ ]:
station = constants.STATIONS[4]
color = "magenta"

str_return, metrics_station, sum_df_station = test_on_station(
    station, color, model, same_color=True
)
str_summary += str_return
sum_dfs_stations.append(sum_df_station)
summary_df.loc[len(summary_df)] = metrics_station

color = constants.COLOR_STATIONS[station]
_, _, _ = test_on_station(station, color, model, same_color=False)

Testing model RELDi
Evaluating for on station HON
Test storm number 89, from 2017-09-06 00:00:00 until 2017-09-15 00:00:00, number of batches: 3168
Shapes of the input tensors: torch.Size([3072, 96, 6]), torch.Size([3072, 96, 5]), torch.Size([3072, 1]), torch.Size([3072, 2, 4]), (3072, 2)
Metrics for SYM_H:
	RMSE: 16.418 | R2: 0.750 | BFE: 24.087 
Metrics for LDi_HON-1h:
	RMSE: 9.055 | R2: 0.924 | BFE: 12.554 | Inside 90%: 0.975
Metrics for LDi_HON-2h:
	RMSE: 17.200 | R2: 0.725 | BFE: 31.393 | Inside 90%: 0.912
Metrics for persistence -1h:
	RMSE: 11.338 | R2: 0.867 | BFE: 14.989 
Metrics for persistence -2h:
	RMSE: 17.664 | R2: 0.677 | BFE: 22.460 
------------------------------------------------
Test storm number 90, from 2017-09-26 00:00:00 until 2017-10-04 00:00:00, number of batches: 2880
Shapes of the input tensors: torch.Size([2784, 96, 6]), torch.Size([2784, 96, 5]), torch.Size([2784, 1]), torch.Size([2784, 2, 4]), (2784, 2)
Metrics for SYM_H:
	RMSE: 13.824 | R2: 0.612 | BFE: 18

,Station,StormIndex,BFE SYM-H,DTW SYM-H,RMSE SYM-H,R2 SYM_H,BFE 1h,BFE 2h,DTW 1h,DTW 2h,RMSE 1h,RMSE 2h,R2 1h,R2 2h,inside 90% 1h,inside 90% 2h
0,HON,89,24.087,13.696,16.418,0.750,12.554000,31.393000,5.634,10.251,9.055,17.200,0.924,0.725,0.975,0.912
1,HON,90,18.365,11.609,13.824,0.612,10.085000,11.990000,5.429,8.726,7.591,11.421,0.883,0.735,0.948,0.858
2,HON,91,21.361,8.706,11.817,0.907,14.090000,21.999001,4.883,7.845,9.426,12.890,0.941,0.890,0.964,0.902
3,HON,92,14.682,7.835,11.334,0.648,12.744000,21.947001,5.459,9.425,8.125,14.463,0.819,0.427,0.926,0.836
4,HON,93,8.774,9.116,10.751,0.751,5.770000,8.345000,4.797,7.972,6.272,10.090,0.915,0.781,0.987,0.950
5,HON,94,15.896,17.129,21.449,-1.051,4.665000,8.138000,4.349,7.346,5.601,9.239,0.860,0.619,0.975,0.844
6,HON,95,14.338,15.281,19.937,0.487,10.180000,13.676000,6.375,9.709,9.293,13.448,0.888,0.766,0.932,0.859
7,HON,96,17.997,12.018,14.699,0.864,17.770000,22.750999,5.693,8.506,9.448,12.864,0.944,0.896,0.937,0.876
8,HON,97,15.667,7.446,10.649,0.942,13.785000,27.476000,5.656,9.011,8.872,14.632,0.960,0.890,0.963,0.895
9,HON,98,14.567,7.381,9.530,0.821,10.093000,12.775000,4.095,6.462,5.989,9.110,0.929,0.836,0.970,0.920


Mean and global MSE output


,Station,StormIndex,BFE SYM-H,DTW SYM-H,RMSE SYM-H,R2 SYM_H,BFE 1h,BFE 2h,DTW 1h,DTW 2h,RMSE 1h,RMSE 2h,R2 1h,R2 2h,inside 90% 1h,inside 90% 2h
21,HON,Mean,15.580,10.554,13.948,0.683,11.226000,17.000000,5.371,8.643,8.291,12.807,0.916,0.787,0.959,0.883
22,HON,Global,29.023,10.578,14.656,0.871,26.721001,32.192001,5.456,8.760,8.922,13.617,0.952,0.889,0.959,0.883


           Station  BFE SYM-H  DTW SYM-H  RMSE SYM-H  R2 SYM_H
StormIndex                                                    
Mean           HON     15.580     10.554      13.948     0.683
Global         HON     29.023     10.578      14.656     0.871
Metrics for persistence -2h:
	RMSE: 10.061 | R2: 0.932 | BFE: 36.434 
Metrics for persistence -2h:
	RMSE: 14.496 | R2: 0.859 | BFE: 45.655 
Diff BFE (SYM-H nowcast - LDi forecast 1h):
Total: 2.302
Diff BFE (SYM-H nowcast - LDi forecast 2h):
Total: -3.169


...

,Station,StormIndex,BFE SYM-H,DTW SYM-H,RMSE SYM-H,R2 SYM_H,BFE 1h,BFE 2h,DTW 1h,DTW 2h,RMSE 1h,RMSE 2h,R2 1h,R2 2h,inside 90% 1h,inside 90% 2h
0,HON,89,24.087,13.696,16.418,0.750,12.554000,31.393000,5.634,10.251,9.055,17.200,0.924,0.725,0.975,0.912
1,HON,90,18.365,11.609,13.824,0.612,10.085000,11.990000,5.429,8.726,7.591,11.421,0.883,0.735,0.948,0.858
2,HON,91,21.361,8.706,11.817,0.907,14.090000,21.999001,4.883,7.845,9.426,12.890,0.941,0.890,0.964,0.902
3,HON,92,14.682,7.835,11.334,0.648,12.744000,21.947001,5.459,9.425,8.125,14.463,0.819,0.427,0.926,0.836
4,HON,93,8.774,9.116,10.751,0.751,5.770000,8.345000,4.797,7.972,6.272,10.090,0.915,0.781,0.987,0.950
5,HON,94,15.896,17.129,21.449,-1.051,4.665000,8.138000,4.349,7.346,5.601,9.239,0.860,0.619,0.975,0.844
6,HON,95,14.338,15.281,19.937,0.487,10.180000,13.676000,6.375,9.709,9.293,13.448,0.888,0.766,0.932,0.859
7,HON,96,17.997,12.018,14.699,0.864,17.770000,22.750999,5.693,8.506,9.448,12.864,0.944,0.896,0.937,0.876
8,HON,97,15.667,7.446,10.649,0.942,13.785000,27.476000,5.656,9.011,8.872,14.632,0.960,0.890,0.963,0.895
9,HON,98,14.567,7.381,9.530,0.821,10.093000,12.775000,4.095,6.462,5.989,9.110,0.929,0.836,0.970,0.920


Mean and global MSE output


,Station,StormIndex,BFE SYM-H,DTW SYM-H,RMSE SYM-H,R2 SYM_H,BFE 1h,BFE 2h,DTW 1h,DTW 2h,RMSE 1h,RMSE 2h,R2 1h,R2 2h,inside 90% 1h,inside 90% 2h
21,HON,Mean,15.580,10.554,13.948,0.683,11.226000,17.000000,5.371,8.643,8.291,12.807,0.916,0.787,0.959,0.883
22,HON,Global,29.023,10.578,14.656,0.871,26.721001,32.192001,5.456,8.760,8.922,13.617,0.952,0.889,0.959,0.883


           Station  BFE SYM-H  DTW SYM-H  RMSE SYM-H  R2 SYM_H
StormIndex                                                    
Mean           HON     15.580     10.554      13.948     0.683
Global         HON     29.023     10.578      14.656     0.871
Metrics for persistence -2h:
	RMSE: 10.061 | R2: 0.932 | BFE: 36.434 
Metrics for persistence -2h:
	RMSE: 14.496 | R2: 0.859 | BFE: 45.655 
Diff BFE (SYM-H nowcast - LDi forecast 1h):
Total: 2.302
Diff BFE (SYM-H nowcast - LDi forecast 2h):
Total: -3.169


,Station,StormIndex,BFE SYM-H,DTW SYM-H,RMSE SYM-H,R2 SYM_H,BFE 1h,BFE 2h,DTW 1h,DTW 2h,RMSE 1h,RMSE 2h,R2 1h,R2 2h,inside 90% 1h,inside 90% 2h
0,HON,89,24.087273,13.695542,16.417782,0.749596,12.554164,31.392675,5.634347,10.250728,9.054785,17.200062,0.923833,0.725165,0.974933,0.911685
1,HON,90,18.365037,11.609193,13.823751,0.611908,10.084810,11.990288,5.429496,8.725995,7.590697,11.421491,0.882984,0.735072,0.948373,0.857701
2,HON,91,21.361243,8.705857,11.817078,0.907304,14.090361,21.999197,4.882601,7.844621,9.425741,12.890362,0.941025,0.889701,0.964425,0.901518
3,HON,92,14.682178,7.834586,11.334126,0.648089,12.743805,21.946768,5.458649,9.424595,8.125154,14.463058,0.819149,0.426971,0.925813,0.835575
4,HON,93,8.773747,9.116456,10.750762,0.751400,5.770063,8.345233,4.797339,7.971769,6.272356,10.090349,0.915378,0.781004,0.987419,0.950108
5,HON,94,15.896121,17.128737,21.449318,-1.050778,4.665092,8.138317,4.348975,7.346022,5.600754,9.239199,0.860175,0.619495,0.975271,0.843818
6,HON,95,14.338431,15.281400,19.936592,0.486716,10.180209,13.675858,6.375165,9.709119,9.292794,13.447620,0.888481,0.766468,0.931739,0.858851
7,HON,96,17.997489,12.018317,14.698519,0.863849,17.770012,22.750555,5.693210,8.505983,9.447601,12.863937,0.943751,0.895715,0.937138,0.875820
8,HON,97,15.666986,7.445808,10.648917,0.941808,13.784632,27.475700,5.655992,9.011390,8.871846,14.632007,0.959609,0.890135,0.962977,0.895102
9,HON,98,14.566696,7.381332,9.529902,0.820954,10.093100,12.775177,4.095369,6.461593,5.989246,9.110369,0.929282,0.836371,0.970065,0.919740


In [ ]:
station = constants.STATIONS[5]
color = "magenta"

str_return, metrics_station, sum_df_station = test_on_station(
    station, color, model, same_color=True
)
str_summary += str_return
sum_dfs_stations.append(sum_df_station)
summary_df.loc[len(summary_df)] = metrics_station

color = constants.COLOR_STATIONS[station]
_, _, _ = test_on_station(station, color, model, same_color=False)

Testing model RELDi
Evaluating for on station SFS
Test storm number 89, from 2017-09-06 00:00:00 until 2017-09-15 00:00:00, number of batches: 3168
Shapes of the input tensors: torch.Size([3072, 96, 6]), torch.Size([3072, 96, 5]), torch.Size([3072, 1]), torch.Size([3072, 2, 4]), (3072, 2)
Metrics for SYM_H:
	RMSE: 22.758 | R2: 0.570 | BFE: 50.584 
Metrics for LDi_SFS-1h:
	RMSE: 16.443 | R2: 0.775 | BFE: 30.366 | Inside 90%: 0.895
Metrics for LDi_SFS-2h:
	RMSE: 22.648 | R2: 0.574 | BFE: 47.286 | Inside 90%: 0.772
Metrics for persistence -1h:
	RMSE: 17.885 | R2: 0.695 | BFE: 25.003 
Metrics for persistence -2h:
	RMSE: 23.706 | R2: 0.465 | BFE: 34.913 
------------------------------------------------
Test storm number 90, from 2017-09-26 00:00:00 until 2017-10-04 00:00:00, number of batches: 2880
Shapes of the input tensors: torch.Size([2784, 96, 6]), torch.Size([2784, 96, 5]), torch.Size([2784, 1]), torch.Size([2784, 2, 4]), (2784, 2)
Metrics for SYM_H:
	RMSE: 18.635 | R2: 0.114 | BFE: 1

,Station,StormIndex,BFE SYM-H,DTW SYM-H,RMSE SYM-H,R2 SYM_H,BFE 1h,BFE 2h,DTW 1h,DTW 2h,RMSE 1h,RMSE 2h,R2 1h,R2 2h,inside 90% 1h,inside 90% 2h
0,SFS,89,50.584,12.795,22.758,0.570,30.365999,47.285999,9.126,13.773,16.443,22.648,0.775,0.574,0.895,0.772
1,SFS,90,18.339,14.217,18.635,0.114,10.038000,12.969000,7.780,11.080,11.153,14.922,0.683,0.432,0.853,0.767
2,SFS,91,20.574,14.574,23.527,0.360,14.848000,22.636000,7.263,11.410,11.551,17.014,0.846,0.665,0.924,0.825
3,SFS,92,27.901,11.100,16.285,0.469,26.235001,32.936001,8.957,12.779,13.471,17.731,0.637,0.371,0.800,0.690
4,SFS,93,21.623,15.902,20.022,0.320,11.698000,19.107000,7.090,10.856,10.177,14.724,0.824,0.632,0.928,0.838
5,SFS,94,19.614,20.399,23.130,-0.127,14.734000,27.121000,5.870,10.256,8.173,14.188,0.859,0.576,0.938,0.807
6,SFS,96,22.693,10.064,17.206,0.672,15.161000,18.219999,7.102,9.703,11.743,14.529,0.847,0.766,0.859,0.806
7,SFS,97,27.028,17.498,26.089,0.302,22.611000,17.808001,8.693,12.191,13.367,16.444,0.817,0.723,0.857,0.692
8,SFS,98,23.734,11.655,15.871,0.168,13.195000,25.593000,6.429,9.741,9.916,14.148,0.675,0.339,0.912,0.800
9,SFS,99,32.981,13.406,19.576,0.747,21.184000,27.330999,8.314,12.253,13.478,18.168,0.880,0.782,0.948,0.878


Mean and global MSE output


,Station,StormIndex,BFE SYM-H,DTW SYM-H,RMSE SYM-H,R2 SYM_H,BFE 1h,BFE 2h,DTW 1h,DTW 2h,RMSE 1h,RMSE 2h,R2 1h,R2 2h,inside 90% 1h,inside 90% 2h
20,SFS,Mean,27.214,14.085,19.772,0.485,19.881001,28.195999,7.776,11.543,12.322,17.362,0.816,0.629,0.889,0.787
21,SFS,Global,54.335,14.339,21.087,0.674,45.664001,56.500000,7.964,11.801,13.368,18.674,0.869,0.744,0.891,0.789


           Station  BFE SYM-H  DTW SYM-H  RMSE SYM-H  R2 SYM_H
StormIndex                                                    
Mean           SFS     27.214     14.085      19.772     0.485
Global         SFS     54.335     14.339      21.087     0.674
Metrics for persistence -2h:
	RMSE: 14.739 | R2: 0.823 | BFE: 52.602 
Metrics for persistence -2h:
	RMSE: 19.692 | R2: 0.685 | BFE: 85.238 
Diff BFE (SYM-H nowcast - LDi forecast 1h):
Total: 8.672
Diff BFE (SYM-H nowcast - LDi forecast 2h):
Total: -2.165


...

,Station,StormIndex,BFE SYM-H,DTW SYM-H,RMSE SYM-H,R2 SYM_H,BFE 1h,BFE 2h,DTW 1h,DTW 2h,RMSE 1h,RMSE 2h,R2 1h,R2 2h,inside 90% 1h,inside 90% 2h
0,SFS,89,50.584,12.795,22.758,0.570,30.365999,47.285999,9.126,13.773,16.443,22.648,0.775,0.574,0.895,0.772
1,SFS,90,18.339,14.217,18.635,0.114,10.038000,12.969000,7.780,11.080,11.153,14.922,0.683,0.432,0.853,0.767
2,SFS,91,20.574,14.574,23.527,0.360,14.848000,22.636000,7.263,11.410,11.551,17.014,0.846,0.665,0.924,0.825
3,SFS,92,27.901,11.100,16.285,0.469,26.235001,32.936001,8.957,12.779,13.471,17.731,0.637,0.371,0.800,0.690
4,SFS,93,21.623,15.902,20.022,0.320,11.698000,19.107000,7.090,10.856,10.177,14.724,0.824,0.632,0.928,0.838
5,SFS,94,19.614,20.399,23.130,-0.127,14.734000,27.121000,5.870,10.256,8.173,14.188,0.859,0.576,0.938,0.807
6,SFS,96,22.693,10.064,17.206,0.672,15.161000,18.219999,7.102,9.703,11.743,14.529,0.847,0.766,0.859,0.806
7,SFS,97,27.028,17.498,26.089,0.302,22.611000,17.808001,8.693,12.191,13.367,16.444,0.817,0.723,0.857,0.692
8,SFS,98,23.734,11.655,15.871,0.168,13.195000,25.593000,6.429,9.741,9.916,14.148,0.675,0.339,0.912,0.800
9,SFS,99,32.981,13.406,19.576,0.747,21.184000,27.330999,8.314,12.253,13.478,18.168,0.880,0.782,0.948,0.878


Mean and global MSE output


,Station,StormIndex,BFE SYM-H,DTW SYM-H,RMSE SYM-H,R2 SYM_H,BFE 1h,BFE 2h,DTW 1h,DTW 2h,RMSE 1h,RMSE 2h,R2 1h,R2 2h,inside 90% 1h,inside 90% 2h
20,SFS,Mean,27.214,14.085,19.772,0.485,19.881001,28.195999,7.776,11.543,12.322,17.362,0.816,0.629,0.889,0.787
21,SFS,Global,54.335,14.339,21.087,0.674,45.664001,56.500000,7.964,11.801,13.368,18.674,0.869,0.744,0.891,0.789


           Station  BFE SYM-H  DTW SYM-H  RMSE SYM-H  R2 SYM_H
StormIndex                                                    
Mean           SFS     27.214     14.085      19.772     0.485
Global         SFS     54.335     14.339      21.087     0.674
Metrics for persistence -2h:
	RMSE: 14.739 | R2: 0.823 | BFE: 52.602 
Metrics for persistence -2h:
	RMSE: 19.692 | R2: 0.685 | BFE: 85.238 
Diff BFE (SYM-H nowcast - LDi forecast 1h):
Total: 8.672
Diff BFE (SYM-H nowcast - LDi forecast 2h):
Total: -2.165


,Station,StormIndex,BFE SYM-H,DTW SYM-H,RMSE SYM-H,R2 SYM_H,BFE 1h,BFE 2h,DTW 1h,DTW 2h,RMSE 1h,RMSE 2h,R2 1h,R2 2h,inside 90% 1h,inside 90% 2h
0,SFS,89,50.583509,12.794882,22.757847,0.569807,30.365574,47.286335,9.126262,13.773238,16.442875,22.647701,0.775427,0.573961,0.894717,0.772464
1,SFS,90,18.338817,14.217015,18.635384,0.114263,10.037889,12.968570,7.780155,11.080265,11.153233,14.922265,0.682729,0.432066,0.853362,0.767462
2,SFS,91,20.574443,14.574017,23.526901,0.359759,14.847600,22.636118,7.262834,11.409846,11.551359,17.014166,0.845659,0.665161,0.924078,0.825163
3,SFS,92,27.901009,11.100243,16.285482,0.469231,26.235157,32.935940,8.957222,12.778894,13.470980,17.730909,0.636836,0.370833,0.800434,0.689805
4,SFS,93,21.622768,15.902404,20.022205,0.319988,11.698040,19.106506,7.089957,10.855685,10.177018,14.724424,0.824315,0.632236,0.928416,0.838178
5,SFS,94,19.613545,20.398529,23.130234,-0.127045,14.734414,27.120859,5.869899,10.255680,8.172853,14.188438,0.859289,0.575917,0.937961,0.807375
6,SFS,96,22.692748,10.063745,17.206263,0.671795,15.160609,18.219549,7.102405,9.703079,11.742761,14.529258,0.847133,0.765977,0.858851,0.806016
7,SFS,97,27.028444,17.497753,26.088697,0.302336,22.611456,17.807711,8.692578,12.191030,13.367096,16.444317,0.816846,0.722813,0.856922,0.691863
8,SFS,98,23.734055,11.655267,15.871263,0.168236,13.195260,25.592720,6.428825,9.740507,9.916247,14.147732,0.675308,0.339077,0.911931,0.800000
9,SFS,99,32.981122,13.405916,19.575520,0.746528,21.184053,27.331356,8.313858,12.252512,13.477529,18.168112,0.879850,0.781665,0.948322,0.878133


In [23]:
summary_df

,Station,Global_BFE_SYM-H,Global_RMSE_SYM-H,Global_R2_SYM-H,Global_RMSE_1h,Global_BFE_1h,Global_R2_1h,Global_inside_90%_1h,Global_RMSE_2h,Global_BFE_2h,Global_R2_2h,Global_inside_90%_2h,RMSE_persistence-1h,BFE_persistence-1h,R2_persistence-1h,RMSE_persistence-2h,BFE_persistence-2h,R2_persistence-2h,Better BFE 1h,Better BFE 2h,Better RMSE 1h,Better RMSE 2h
0,ABG,44.530046,19.620636,0.830243,11.128581,25.633680,0.945389,0.911835,17.194424,40.916145,0.869631,0.796649,13.091836,36.519421,0.915998,19.043194,52.785847,0.822267,19/20,14/20,20/20,13/20
1,MMB,48.868305,21.841389,0.648062,12.723472,51.677444,0.880569,0.926617,17.774429,77.944939,0.766924,0.841804,13.975657,63.204327,0.839810,18.071266,73.889000,0.732165,19/21,9/21,20/21,19/21
2,CLF,73.669954,28.087835,0.064327,12.652555,64.437164,0.810135,0.933223,17.307835,74.647003,0.644718,0.858612,13.114584,66.488892,0.773754,17.000250,100.730858,0.619825,23/23,14/23,23/23,23/23
3,TUC,38.910315,21.551609,0.617346,12.726727,37.391785,0.866562,0.921512,16.995157,37.349285,0.762043,0.831022,13.454135,46.375568,0.834477,17.293234,54.432415,0.726537,19/23,14/23,22/23,19/23
4,HON,29.023123,14.656344,0.871474,8.921650,26.721169,0.952376,0.958948,13.616827,32.191723,0.889059,0.882958,10.060527,36.433723,0.932309,14.495803,45.654743,0.859469,22/23,12/23,23/23,14/23
5,SFS,54.335281,21.086529,0.674201,13.367838,45.663631,0.869063,0.890687,18.673998,56.499844,0.744486,0.789451,14.738920,52.602409,0.823343,19.692383,85.238220,0.684648,21/22,10/22,22/22,19/22


In [24]:
print("SYM-H metrics")
print(f'BFE: {summary_df[[f"Global_BFE_SYM-H"]].mean().sum():.3f}')
print(f'RMSE: {summary_df[[f"Global_RMSE_SYM-H"]].mean().sum():.3f}')
print("Forecast metrics")
for forecast_step in range(FORECAST_STEPS):
    print(
        f'BFE {forecast_step + 1}h: {summary_df[[f"Global_BFE_{forecast_step + 1}h"]].mean().sum():.3f}'
    )
    print(
        f'RMSE {forecast_step + 1}h: {summary_df[[f"Global_RMSE_{forecast_step + 1}h"]].mean().sum():.3f}'
    )

SYM-H metrics
BFE: 48.223
RMSE: 21.141
Forecast metrics
BFE 1h: 41.921
RMSE 1h: 11.920
BFE 2h: 53.258
RMSE 2h: 16.927


In [25]:
str_metrics = ""

for forecast_step in range(FORECAST_STEPS):
    str_metrics += f'BFE {forecast_step + 1}h: {summary_df[[f"Global_BFE_{forecast_step + 1}h"]].mean().sum():.3f}\nRMSE {forecast_step + 1}h: {summary_df[["Global_RMSE_1h"]].mean().sum():.3f}\n'

for forecast_step in range(FORECAST_STEPS):
    summary_df[f"Diff_{forecast_step + 1}h"] = (
        summary_df[f"Global_BFE_{forecast_step + 1}h"] - summary_df["Global_BFE_SYM-H"]
    )
    summary_df[f"Diff_RMSE_{forecast_step + 1}h"] = (
        summary_df[f"Global_RMSE_{forecast_step + 1}h"]
        - summary_df["Global_RMSE_SYM-H"]
    )
    str_metrics += f"Differences BFE {forecast_step + 1}h\n"
    print(f"Differences BFE {forecast_step + 1}h")
    str_metrics += (
        summary_df.set_index("Station")[f"Diff_{forecast_step + 1}h"]
        .round(3)
        .to_string()
    )
    print(
        summary_df.set_index("Station")[f"Diff_{forecast_step + 1}h"]
        .round(3)
        .to_string()
    )
    str_metrics += f"\nSum differences: {summary_df.set_index('Station')[f'Diff_{forecast_step + 1}h'].sum():.3f}\n"
    print(
        f"Sum differences {forecast_step + 1}h: {summary_df.set_index('Station')[f'Diff_{forecast_step + 1}h'].sum():.3f}"
    )

    str_metrics += f"Differences RMSE {forecast_step + 1}h\n"
    print(f"Differences RMSE {forecast_step + 1}h")
    str_metrics += (
        summary_df.set_index("Station")[f"Diff_RMSE_{forecast_step + 1}h"]
        .round(3)
        .to_string()
    )
    print(
        summary_df.set_index("Station")[f"Diff_RMSE_{forecast_step + 1}h"]
        .round(3)
        .to_string()
    )
    str_metrics += f"\nSum differences RMSE: {summary_df.set_index('Station')[f'Diff_RMSE_{forecast_step + 1}h'].sum():.3f}\n"
    print(
        f"Sum differences RMSE {forecast_step + 1}h: {summary_df.set_index('Station')[f'Diff_RMSE_{forecast_step + 1}h'].sum():.3f}"
    )

Differences BFE 1h
Station
ABG   -18.896
MMB     2.809
CLF    -9.233
TUC    -1.519
HON    -2.302
SFS    -8.672
Sum differences 1h: -37.812
Differences RMSE 1h
Station
ABG    -8.492
MMB    -9.118
CLF   -15.435
TUC    -8.825
HON    -5.735
SFS    -7.719
Sum differences RMSE 1h: -55.324
Differences BFE 2h
Station
ABG    -3.614
MMB    29.077
CLF     0.977
TUC    -1.561
HON     3.169
SFS     2.165
Sum differences 2h: 30.212
Differences RMSE 2h
Station
ABG    -2.426
MMB    -4.067
CLF   -10.780
TUC    -4.556
HON    -1.040
SFS    -2.413
Sum differences RMSE 2h: -25.282


In [26]:
cols_print = []
for forecast_step in range(FORECAST_STEPS):
    cols_print.append(f"Diff_{forecast_step + 1}h")
    cols_print.append(f"Better BFE {forecast_step + 1}h")
    cols_print.append(f"Diff_RMSE_{forecast_step + 1}h")
    cols_print.append(f"Better RMSE {forecast_step + 1}h")
print(summary_df.set_index("Station")[cols_print])

           Diff_1h Better BFE 1h  Diff_RMSE_1h Better RMSE 1h    Diff_2h  \
Station                                                                    
ABG     -18.896365         19/20     -8.492055          20/20  -3.613900   
MMB       2.809139         19/21     -9.117917          20/21  29.076634   
CLF      -9.232790         23/23    -15.435281          23/23   0.977049   
TUC      -1.518530         19/23     -8.824883          22/23  -1.561030   
HON      -2.301954         22/23     -5.734694          23/23   3.168600   
SFS      -8.671649         21/22     -7.718691          22/22   2.164563   

        Better BFE 2h  Diff_RMSE_2h Better RMSE 2h  
Station                                             
ABG             14/20     -2.426212          13/20  
MMB              9/21     -4.066959          19/21  
CLF             14/23    -10.780001          23/23  
TUC             14/23     -4.556452          19/23  
HON             12/23     -1.039517          14/23  
SFS             10/2

In [27]:
print(str_metrics)

BFE 1h: 41.921
RMSE 1h: 11.920
BFE 2h: 53.258
RMSE 2h: 11.920
Differences BFE 1h
Station
ABG   -18.896
MMB     2.809
CLF    -9.233
TUC    -1.519
HON    -2.302
SFS    -8.672
Sum differences: -37.812
Differences RMSE 1h
Station
ABG    -8.492
MMB    -9.118
CLF   -15.435
TUC    -8.825
HON    -5.735
SFS    -7.719
Sum differences RMSE: -55.324
Differences BFE 2h
Station
ABG    -3.614
MMB    29.077
CLF     0.977
TUC    -1.561
HON     3.169
SFS     2.165
Sum differences: 30.212
Differences RMSE 2h
Station
ABG    -2.426
MMB    -4.067
CLF   -10.780
TUC    -4.556
HON    -1.040
SFS    -2.413
Sum differences RMSE: -25.282



In [28]:
print(str_summary)

Information for station ABG
   StormIndex  BFE SYM-H  RMSE SYM-H     BFE 1h     BFE 2h    RMSE 1h  \
18       Mean  29.870066   18.580623  15.744441  24.489235  10.462987   
19     Global  44.530046   19.620636  25.633680  40.916145  11.128581   

      RMSE 2h  
18  16.103891  
19  17.194424  
Metrics for persistence LDi_ABG-1h:
	RMSE: 13.092 | BFE: 36.519
Metrics for persistence LDi_ABG-2h:
	RMSE: 19.043 | BFE: 52.786

Information for station MMB
   StormIndex  BFE SYM-H  RMSE SYM-H     BFE 1h     BFE 2h    RMSE 1h  \
19       Mean  25.250677   20.798985  19.461824  27.849573  11.534310   
20     Global  48.868305   21.841389  51.677444  77.944939  12.723472   

      RMSE 2h  
19  16.166386  
20  17.774429  
Metrics for persistence LDi_MMB-1h:
	RMSE: 13.976 | BFE: 63.204
Metrics for persistence LDi_MMB-2h:
	RMSE: 18.071 | BFE: 73.889

Information for station CLF
   StormIndex  BFE SYM-H  RMSE SYM-H     BFE 1h     BFE 2h    RMSE 1h  \
21       Mean  32.105841   25.317692  20.372385  

In [29]:
import os
import constants

for station in constants.STATIONS:
    os.makedirs(f"./figs/{station}/forecasts", exist_ok=True)
    os.makedirs(f"./figs/{station}/bfe", exist_ok=True)
    os.makedirs(f"./figs/{station}/grids", exist_ok=True)

    os.makedirs(f"./figs_colors/{station}/forecasts", exist_ok=True)
    os.makedirs(f"./figs_colors/{station}/bfe", exist_ok=True)
    os.makedirs(f"./figs_colors/{station}/grids", exist_ok=True)

In [30]:
from PIL import Image
import os


def plot_grid(images_paths, output_path, rows, columns):
    fig, axes = plt.subplots(rows, columns, figsize=(13 * columns, 7 * rows))

    for idx, image_path in enumerate(images_paths):
        row, col = divmod(idx, columns)
        if os.path.exists(image_path):
            img = Image.open(image_path)
            axes[row, col].imshow(img)
            # axes[row, col].set_title(f"{hour + 1} ahead")
        else:
            print(f"Image not found: {image_path}")
            plt.close()
            return
        axes[row, col].axis("off")

    # plt.tight_layout(rect=[0, 0, 1, 1])
    plt.tight_layout(
        pad=0.01,  # Overall padding around the figure (default is 1.08)
        w_pad=0.01,  # Padding between subplots in width (default is 0.2)
        h_pad=0.01,  # Padding between subplots in height (default is 0.2)
    )
    plt.savefig(output_path, dpi=150, transparent=True, bbox_inches="tight")
    plt.close()


def plot_all_grids(same_color=False):

    plot_grid(
        [
            f"./figs{"_colors" if same_color else ""}/{station}/bfe/bfe-comparison-sym-ldi-{station}-LDi_{station}-1h.png"
            for station in constants.STATIONS
        ],
        f"./figs{"_colors" if same_color else ""}/grids/bfe-sym-diff-ldi-stations-1h_grid.png",
        rows=3,
        columns=2,
    )

    plot_grid(
        [
            f"./figs{"_colors" if same_color else ""}/{station}/bfe/bfe-ldi-{station}-LDi_{station}-1h.png"
            for station in constants.STATIONS
        ],
        f"./figs{"_colors" if same_color else ""}/grids/bfe-ldi-stations-1h_grid.png",
        rows=3,
        columns=2,
    )

    plot_grid(
        [
            f"./figs{"_colors" if same_color else ""}/{station}/bfe/bfe-comparison-sym-ldi-{station}-LDi_{station}-2h.png"
            for station in constants.STATIONS
        ],
        f"./figs{"_colors" if same_color else ""}/grids/bfe-sym-diff-ldi-stations-2h_grid.png",
        rows=3,
        columns=2,
    )

    plot_grid(
        [
            f"./figs{"_colors" if same_color else ""}/{station}/bfe/bfe-ldi-{station}-LDi_{station}-2h.png"
            for station in constants.STATIONS
        ],
        f"./figs{"_colors" if same_color else ""}/grids/bfe-ldi-stations-2h_grid.png",
        rows=3,
        columns=2,
    )

    for station in constants.STATIONS:
        for start_date, end_date, storm_index in storm_dates.TEST_STORMS:
            plot_grid(
                [
                    f"./figs{"_colors" if same_color else ""}/{station}/forecasts/forecast-ldi-{station}-LDi_{station}-1h-test-storm-{storm_index}.png",
                    f"./figs{"_colors" if same_color else ""}/{station}/bfe/bfe-ldi-{station}-LDi_{station}-1h-test-storm-{storm_index}.png",
                    f"./figs{"_colors" if same_color else ""}/{station}/forecasts/forecast-ldi-{station}-LDi_{station}-2h-test-storm-{storm_index}.png",
                    f"./figs{"_colors" if same_color else ""}/{station}/bfe/bfe-ldi-{station}-LDi_{station}-2h-test-storm-{storm_index}.png",
                ],
                f"./figs{"_colors" if same_color else ""}/{station}/grids/grid-{station}-test-storm-{storm_index}.png",
                rows=2,
                columns=2,
            )

        plot_grid(
            [
                f"./figs{"_colors" if same_color else ""}/{station}/forecasts/forecast-ldi-{station}-LDi_{station}-1h-test-storm-104-may-storm.png",
                f"./figs{"_colors" if same_color else ""}/{station}/bfe/bfe-ldi-{station}-LDi_{station}-1h-test-storm-104-may-storm.png",
                f"./figs{"_colors" if same_color else ""}/{station}/forecasts/forecast-ldi-{station}-LDi_{station}-2h-test-storm-104-may-storm.png",
                f"./figs{"_colors" if same_color else ""}/{station}/bfe/bfe-ldi-{station}-LDi_{station}-2h-test-storm-104-may-storm.png",
            ],
            f"./figs{"_colors" if same_color else ""}/{station}/grids/grid-{station}-test-storm-104-may-peak.png",
            rows=2,
            columns=2,
        )

        plot_grid(
            [
                f"./figs{"_colors" if same_color else ""}/{station}/forecasts/forecast-ldi-{station}-LDi_{station}-1h-test-storm-108-october-storm.png",
                f"./figs{"_colors" if same_color else ""}/{station}/bfe/bfe-ldi-{station}-LDi_{station}-1h-test-storm-108-october-storm.png",
                f"./figs{"_colors" if same_color else ""}/{station}/forecasts/forecast-ldi-{station}-LDi_{station}-2h-test-storm-108-october-storm.png",
                f"./figs{"_colors" if same_color else ""}/{station}/bfe/bfe-ldi-{station}-LDi_{station}-2h-test-storm-108-october-storm.png",
            ],
            f"./figs{"_colors" if same_color else ""}/{station}/grids/grid-{station}-test-storm-108-october-peak.png",
            rows=2,
            columns=2,
        )

    for station in constants.STATIONS:
        plot_grid(
            [
                f"./figs{"_colors" if same_color else ""}/{station}/forecasts/forecast-ldi-{station}-LDi_{station}-1h-test-storm-104-may-storm.png",
                f"./figs{"_colors" if same_color else ""}/{station}/forecasts/persistence-ldi-{station}-1h-test-storm-104-may-storm.png",
                f"./figs{"_colors" if same_color else ""}/{station}/forecasts/forecast-ldi-{station}-LDi_{station}-2h-test-storm-104-may-storm.png",
                f"./figs{"_colors" if same_color else ""}/{station}/forecasts/persistence-ldi-{station}-2h-test-storm-104-may-storm.png",
            ],
            f"./figs{"_colors" if same_color else ""}/{station}/grids/persistence-grid-{station}-test-storm-104-may.png",
            rows=2,
            columns=2,
        )


plot_all_grids(same_color=False)
plot_all_grids(same_color=True)

Image not found: ./figs/ABG/forecasts/forecast-ldi-ABG-LDi_ABG-1h-test-storm-93.png
Image not found: ./figs/ABG/forecasts/forecast-ldi-ABG-LDi_ABG-1h-test-storm-103.png
Image not found: ./figs/ABG/forecasts/forecast-ldi-ABG-LDi_ABG-1h-test-storm-109.png
Image not found: ./figs/MMB/forecasts/forecast-ldi-MMB-LDi_MMB-1h-test-storm-93.png
Image not found: ./figs/MMB/forecasts/forecast-ldi-MMB-LDi_MMB-1h-test-storm-94.png
Image not found: ./figs/SFS/forecasts/forecast-ldi-SFS-LDi_SFS-1h-test-storm-95.png
Image not found: ./figs_colors/ABG/forecasts/forecast-ldi-ABG-LDi_ABG-1h-test-storm-93.png
Image not found: ./figs_colors/ABG/forecasts/forecast-ldi-ABG-LDi_ABG-1h-test-storm-103.png
Image not found: ./figs_colors/ABG/forecasts/forecast-ldi-ABG-LDi_ABG-1h-test-storm-109.png
Image not found: ./figs_colors/MMB/forecasts/forecast-ldi-MMB-LDi_MMB-1h-test-storm-93.png
Image not found: ./figs_colors/MMB/forecasts/forecast-ldi-MMB-LDi_MMB-1h-test-storm-94.png
Image not found: ./figs_colors/SFS/fo